# 01 DATA AND CROPS

Extracted from the original notebooks. **Outputs preserved.** Originals are unmodified.


## A · Raw data assembly


**`BCF` cell 5** — CELL 4: Count Image Types from SeriesDescription Column  
<sub>1 output block(s) preserved</sub>


In [6]:
# CELL 4: Count Image Types from SeriesDescription Column
print(
    dicom_info["SeriesDescription"]
    .value_counts()
)

SeriesDescription
cropped images           3567
ROI mask images          3247
full mammogram images    2857
Name: count, dtype: int64


**`BCF` cell 6** — CELL 5: Split Dataset into Full Mammogram / Cropped / ROI Mask Subsets  
<sub>1 output block(s) preserved</sub>


In [7]:
# CELL 5: Split Dataset into Full Mammogram / Cropped / ROI Mask Subsets
full_images = dicom_info[
    dicom_info["SeriesDescription"]
    .str.contains("full", case=False, na=False)
].copy()

cropped_images = dicom_info[
    dicom_info["SeriesDescription"]
    .str.contains("cropped", case=False, na=False)
].copy()

roi_masks = dicom_info[
    dicom_info["SeriesDescription"]
    .str.contains("mask", case=False, na=False)
].copy()

print("Full mammograms :", len(full_images))
print("Cropped images  :", len(cropped_images))
print("ROI masks       :", len(roi_masks))

Full mammograms : 2857
Cropped images  : 3567
ROI masks       : 3247


**`BCF` cell 13** — CELL 13: Attach BENIGN/MALIGNANT labels to cropped images  
<sub>1 output block(s) preserved</sub>


In [23]:
# CELL 13: Attach BENIGN/MALIGNANT labels to cropped images
mass_train = all_csvs['mass_case_description_train_set.csv']
mass_test  = all_csvs['mass_case_description_test_set.csv']
calc_train = all_csvs['calc_case_description_train_set.csv']
calc_test  = all_csvs['calc_case_description_test_set.csv']

def make_binary(df, abn_type, split):
    df = df.copy()

    # Mass CSV uses breast_density; calc CSV uses breast density.
    if 'breast_density' not in df.columns and 'breast density' in df.columns:
        df['breast_density'] = df['breast density']

    pathology_upper = df['pathology'].astype(str).str.upper()
    df['label'] = pathology_upper.apply(lambda x: 1 if 'MALIGNANT' in x else 0)
    df['label_name'] = df['label'].map({0: 'BENIGN', 1: 'MALIGNANT'})
    df['abn_type'] = abn_type
    df['split'] = split
    return df

all_labels = pd.concat([
    make_binary(mass_train, 'mass', 'train'),
    make_binary(mass_test, 'mass', 'test'),
    make_binary(calc_train, 'calcification', 'train'),
    make_binary(calc_test, 'calcification', 'test'),
], ignore_index=True)

print('All label counts:')
print(all_labels['label_name'].value_counts())
print('\nTrain:')
print(all_labels[all_labels['split'] == 'train']['label_name'].value_counts())
print('\nTest:')
print(all_labels[all_labels['split'] == 'test']['label_name'].value_counts())
print('\nOriginal pathology values:')
print(all_labels['pathology'].value_counts())
print('\nColumns in label file:')
print(all_labels.columns.tolist())

All label counts:
label_name
BENIGN       2111
MALIGNANT    1457
Name: count, dtype: int64

Train:
label_name
BENIGN       1683
MALIGNANT    1181
Name: count, dtype: int64

Test:
label_name
BENIGN       428
MALIGNANT    276
Name: count, dtype: int64

Original pathology values:
pathology
MALIGNANT                  1457
BENIGN                     1429
BENIGN_WITHOUT_CALLBACK     682
Name: count, dtype: int64

Columns in label file:
['patient_id', 'breast_density', 'left or right breast', 'image view', 'abnormality id', 'abnormality type', 'mass shape', 'mass margins', 'assessment', 'pathology', 'subtlety', 'image file path', 'cropped image file path', 'ROI mask file path', 'label', 'label_name', 'abn_type', 'split', 'breast density', 'calc type', 'calc distribution']


**`BCF` cell 15** — CELL 15 - Build exact matching key from the label path  
<sub>1 output block(s) preserved</sub>


In [25]:
# CELL 15 - Build exact matching key from the label path
# The first folder in "cropped image file path" matches dicom_info PatientID.

def get_case_folder(cropped_dcm_path):
    if not isinstance(cropped_dcm_path, str):
        return None
    return cropped_dcm_path.strip().replace('\\', '/').split('/')[0]

all_labels['dicom_patient_id'] = all_labels['cropped image file path'].apply(get_case_folder)

label_case_ids = set(all_labels['dicom_patient_id'].dropna())
dicom_case_ids = set(cropped_images['PatientID'].dropna())

print(f'Unique label cropped-case IDs : {len(label_case_ids)}')
print(f'Unique cropped image PatientIDs: {len(dicom_case_ids)}')
print(f'Overlap                       : {len(label_case_ids & dicom_case_ids)}')
print(f'Label IDs missing from dicom   : {len(label_case_ids - dicom_case_ids)}')

if len(label_case_ids - dicom_case_ids) > 0:
    print('Missing examples:')
    print(list(label_case_ids - dicom_case_ids)[:10])


Unique label cropped-case IDs : 3568
Unique cropped image PatientIDs: 3567
Overlap                       : 3567
Label IDs missing from dicom   : 1
Missing examples:
['Calc-Training_P_01563_RIGHT_MLO_2']


**`BCF` cell 16** — CELL 16: Build Full Matching Table (Labels + Full/Crop/Mask Image Paths)  
<sub>1 output block(s) preserved</sub>


In [26]:
# CELL 16: Build Full Matching Table (Labels + Full/Crop/Mask Image Paths)
import os
import pandas as pd

# UPDATED AutoDL GPU root path (replaced Colab Drive path)
DATA_ROOT = "/root/autodl-tmp/CBIS"
os.makedirs(DATA_ROOT, exist_ok=True)

# FULL DATASET MATCHING TABLE
# Match each label row with its full mammogram, cropped image, and ROI mask.
# Run this after all_labels, full_images, cropped_images, and roi_masks exist.

jpeg_dir = os.path.join(DATA_ROOT, "jpeg")

def get_case_folder(path):
    if not isinstance(path, str):
        return None
    return path.strip().replace("\\", "/").split("/")[0]

def get_drive_jpeg_path(image_path):
    if not isinstance(image_path, str):
        return None
    return image_path.replace("CBIS-DDSM/jpeg", jpeg_dir).replace("\\", "/")

# Make sure dicom_info has usable full paths and file-existence flags.
if "full_path" not in dicom_info.columns:
    dicom_info["full_path"] = dicom_info["image_path"].apply(get_drive_jpeg_path)

if "file_exists" not in dicom_info.columns:
    dicom_info["file_exists"] = dicom_info["full_path"].apply(os.path.exists)

# Rebuild image groups after full_path and file_exists exist.
full_images = dicom_info[
    dicom_info["SeriesDescription"].str.contains("full", case=False, na=False)
].copy()

cropped_images = dicom_info[
    dicom_info["SeriesDescription"].str.contains("cropped", case=False, na=False)
].copy()

roi_masks = dicom_info[
    dicom_info["SeriesDescription"].str.contains("mask", case=False, na=False)
].copy()

# Create exact DICOM case IDs from label CSV paths.
all_labels = all_labels.copy()
all_labels["full_case_id"] = all_labels["image file path"].apply(get_case_folder)
all_labels["cropped_case_id"] = all_labels["cropped image file path"].apply(get_case_folder)
all_labels["roi_case_id"] = all_labels["ROI mask file path"].apply(get_case_folder)

# Prepare lookup tables.
full_lookup = full_images.copy()
crop_lookup = cropped_images.copy()
roi_lookup = roi_masks.copy()

full_lookup["full_jpeg_path"] = full_lookup["image_path"].apply(get_drive_jpeg_path)
crop_lookup["cropped_jpeg_path"] = crop_lookup["image_path"].apply(get_drive_jpeg_path)
roi_lookup["roi_mask_jpeg_path"] = roi_lookup["image_path"].apply(get_drive_jpeg_path)

full_lookup = full_lookup[
    ["PatientID", "full_jpeg_path", "SeriesInstanceUID", "file_exists"]
].rename(columns={
    "PatientID": "full_case_id",
    "SeriesInstanceUID": "full_series_uid",
    "file_exists": "full_file_exists",
})

crop_lookup = crop_lookup[
    ["PatientID", "cropped_jpeg_path", "SeriesInstanceUID", "file_exists"]
].rename(columns={
    "PatientID": "cropped_case_id",
    "SeriesInstanceUID": "cropped_series_uid",
    "file_exists": "cropped_file_exists",
})

roi_lookup = roi_lookup[
    ["PatientID", "roi_mask_jpeg_path", "SeriesInstanceUID", "file_exists"]
].rename(columns={
    "PatientID": "roi_case_id",
    "SeriesInstanceUID": "roi_series_uid",
    "file_exists": "roi_file_exists",
})

# Merge everything into one checking table.
dataset_match_table = all_labels.merge(
    full_lookup,
    on="full_case_id",
    how="left"
).merge(
    crop_lookup,
    on="cropped_case_id",
    how="left"
).merge(
    roi_lookup,
    on="roi_case_id",
    how="left"
)

# Add availability flags.
dataset_match_table["has_full_mammogram"] = dataset_match_table["full_jpeg_path"].notna()
dataset_match_table["has_cropped_image"] = dataset_match_table["cropped_jpeg_path"].notna()
dataset_match_table["has_roi_mask"] = dataset_match_table["roi_mask_jpeg_path"].notna()

check_cols = [
    "patient_id",
    "full_case_id",
    "cropped_case_id",
    "roi_case_id",
    "pathology",
    "label_name",
    "abn_type",
    "split",
    "left or right breast",
    "image view",
    "has_full_mammogram",
    "has_cropped_image",
    "has_roi_mask",
    "full_file_exists",
    "cropped_file_exists",
    "roi_file_exists",
    "full_jpeg_path",
    "cropped_jpeg_path",
    "roi_mask_jpeg_path",
]

dataset_match_table = dataset_match_table[check_cols]

print("Dataset matching summary")
print("=" * 60)
print(f"Total label rows       : {len(dataset_match_table)}")
print(f"Has full mammogram     : {dataset_match_table['has_full_mammogram'].sum()}")
print(f"Has cropped image      : {dataset_match_table['has_cropped_image'].sum()}")
print(f"Has ROI mask           : {dataset_match_table['has_roi_mask'].sum()}")

print("\nMissing counts")
print("=" * 60)
print(f"Missing full mammogram : {(~dataset_match_table['has_full_mammogram']).sum()}")
print(f"Missing cropped image  : {(~dataset_match_table['has_cropped_image']).sum()}")
print(f"Missing ROI mask       : {(~dataset_match_table['has_roi_mask']).sum()}")

print("\nMissing by split")
print("=" * 60)
print(
    dataset_match_table
    .assign(
        missing_full=~dataset_match_table["has_full_mammogram"],
        missing_crop=~dataset_match_table["has_cropped_image"],
        missing_roi=~dataset_match_table["has_roi_mask"],
    )
    .groupby("split")[["missing_full", "missing_crop", "missing_roi"]]
    .sum()
)

print("\nMissing by cancer type / abnormality type")
print("=" * 60)
print(
    dataset_match_table
    .assign(
        missing_full=~dataset_match_table["has_full_mammogram"],
        missing_crop=~dataset_match_table["has_cropped_image"],
        missing_roi=~dataset_match_table["has_roi_mask"],
    )
    .groupby(["label_name", "abn_type"])[["missing_full", "missing_crop", "missing_roi"]]
    .sum()
)

missing_any = dataset_match_table[
    (~dataset_match_table["has_full_mammogram"]) |
    (~dataset_match_table["has_cropped_image"]) |
    (~dataset_match_table["has_roi_mask"])
].copy()

print("\nRows with anything missing:")
print(missing_any)

dataset_match_table.to_csv(f"{DATA_ROOT}/dataset_match_table.csv", index=False)
missing_any.to_csv(f"{DATA_ROOT}/dataset_missing_rows.csv", index=False)

print("\nSaved CSV files to CBIS folder:")
print(f"{DATA_ROOT}/dataset_match_table.csv")
print(f"{DATA_ROOT}/dataset_missing_rows.csv")


Dataset matching summary
Total label rows       : 3568
Has full mammogram     : 3286
Has cropped image      : 3567
Has ROI mask           : 3247

Missing counts
Missing full mammogram : 282
Missing cropped image  : 1
Missing ROI mask       : 321

Missing by split
       missing_full  missing_crop  missing_roi
split                                         
test            282             0          320
train             0             1            1

Missing by cancer type / abnormality type
                          missing_full  missing_crop  missing_roi
label_name abn_type                                              
BENIGN     calcification           180             1          194
           mass                      0             0            0
MALIGNANT  calcification           102             0          127
           mass                      0             0            0

Rows with anything missing:
     patient_id                     full_case_id  \
2912    P_01563  Calc-Traini

**`BCF` cell 17** — CELL 17 — Extract P_XXXXX from dicom_info PatientID and merge with labels  
<sub>1 output block(s) preserved</sub>


In [27]:
# CELL 17 — Extract P_XXXXX from dicom_info PatientID and merge with labels

# Step 1: Extract P_XXXXX from PatientID like 'Mass-Training_P_01265_RIGHT_MLO_1'
def extract_pid(full_id):
    if pd.isna(full_id): return None
    parts = str(full_id).split('_')
    for i, p in enumerate(parts):
        if p == 'P' and i+1 < len(parts):
            return f'P_{parts[i+1]}'
    return None

cropped_images['pid'] = cropped_images['PatientID'].apply(extract_pid)

# Step 2: Also extract laterality and view from PatientID
# PatientID = 'Mass-Training_P_01265_RIGHT_MLO_1'
def extract_lat(full_id):
    s = str(full_id).upper()
    if 'LEFT'  in s: return 'LEFT'
    if 'RIGHT' in s: return 'RIGHT'
    return 'UNKNOWN'

def extract_view(full_id):
    s = str(full_id).upper()
    if '_CC'  in s: return 'CC'
    if '_MLO' in s: return 'MLO'
    return 'UNKNOWN'

cropped_images['laterality'] = cropped_images['PatientID'].apply(extract_lat)
cropped_images['view']       = cropped_images['PatientID'].apply(extract_view)

# Step 3: Verify extraction worked
print('Sample extracted values:')
print(cropped_images[['PatientID','pid','laterality','view']].head(10).to_string())
print()

# Step 4: Check overlap now
dicom_pids = set(cropped_images['pid'].dropna().unique())
label_pids = set(all_labels['patient_id'].dropna().unique())
overlap    = dicom_pids & label_pids

print(f'Unique PIDs in dicom_info (extracted) : {len(dicom_pids)}')
print(f'Unique PIDs in labels                 : {len(label_pids)}')
print(f'Overlap                               : {len(overlap)}')
print(f'NOT matched                           : {len(label_pids - dicom_pids)}')

Sample extracted values:
                            PatientID      pid laterality view
0   Mass-Training_P_01265_RIGHT_MLO_1  P_01265      RIGHT  MLO
3         Calc-Test_P_00562_LEFT_CC_2  P_00562       LEFT   CC
6    Calc-Training_P_00181_RIGHT_CC_1  P_00181      RIGHT   CC
7     Calc-Training_P_01015_LEFT_CC_1  P_01015       LEFT   CC
10    Calc-Training_P_01497_LEFT_CC_1  P_01497       LEFT   CC
13   Mass-Training_P_00242_RIGHT_CC_1  P_00242      RIGHT   CC
19  Mass-Training_P_00634_RIGHT_MLO_1  P_00634      RIGHT  MLO
21       Mass-Test_P_00882_RIGHT_CC_1  P_00882      RIGHT   CC
26       Calc-Test_P_00857_RIGHT_CC_1  P_00857      RIGHT   CC
28   Calc-Training_P_01338_LEFT_MLO_1  P_01338       LEFT  MLO

Unique PIDs in dicom_info (extracted) : 1566
Unique PIDs in labels                 : 1566
Overlap                               : 1566
NOT matched                           : 0


**`BCF` cell 18** — CELL 18 - Build final dataset with labels attached using exact cropped-image keys  
<sub>1 output block(s) preserved</sub>


In [28]:
# CELL 18 - Build final dataset with labels attached using exact cropped-image keys

# Make sure the exact label key exists.
if 'dicom_patient_id' not in all_labels.columns:
    def get_case_folder(cropped_dcm_path):
        if not isinstance(cropped_dcm_path, str):
            return None
        return cropped_dcm_path.strip().replace('\\', '/').split('/')[0]
    all_labels['dicom_patient_id'] = all_labels['cropped image file path'].apply(get_case_folder)

# One cropped image row per DICOM PatientID/case folder.
cropped_lookup = cropped_images.drop_duplicates('PatientID').set_index('PatientID')

records = []
unmatched = []

for _, lrow in all_labels.iterrows():
    case_id = lrow['dicom_patient_id']

    if case_id not in cropped_lookup.index:
        unmatched.append(lrow.to_dict())
        continue

    img_row = cropped_lookup.loc[case_id]

    records.append({
        'image_path': img_row['full_path'],
        'dicom_patient_id': case_id,
        'patient_id': lrow['patient_id'],
        'laterality': lrow.get('left or right breast', None),
        'view': lrow.get('image view', None),
        'label': int(lrow['label']),
        'label_name': lrow['label_name'],
        'pathology': lrow['pathology'],
        'abn_type': lrow['abn_type'],
        'split': lrow['split'],
        'breast_density': lrow.get('breast_density', None),
        'assessment': lrow.get('assessment', None),
        'subtlety': lrow.get('subtlety', None),
        'cropped_dicom_path': lrow.get('cropped image file path', None),
        'roi_mask_dicom_path': lrow.get('ROI mask file path', None),
    })

final_df = pd.DataFrame(records)
unmatched_df = pd.DataFrame(unmatched)

train_df = final_df[final_df['split'] == 'train'].reset_index(drop=True)
test_df  = final_df[final_df['split'] == 'test'].reset_index(drop=True)

print(f'Total labels        : {len(all_labels)}')
print(f'Total exact matched : {len(final_df)}')
print(f'Unmatched labels    : {len(unmatched_df)}')
print(f'Train               : {len(train_df)}')
print(f'Test                : {len(test_df)}')

print('\nTrain labels:')
print(train_df['label_name'].value_counts())
print('\nTest labels:')
print(test_df['label_name'].value_counts())
print('\nTrain by type:')
print(train_df.groupby(['abn_type', 'label_name']).size())

train_df.to_csv(f'{DATA_ROOT}/cbis_train.csv', index=False)
test_df.to_csv(f'{DATA_ROOT}/cbis_test.csv', index=False)
final_df.to_csv(f'{DATA_ROOT}/cbis_all_exact_matched.csv', index=False)

if len(unmatched_df) > 0:
    unmatched_df.to_csv(f'{DATA_ROOT}/cbis_unmatched_labels.csv', index=False)
    print(f'\nSaved unmatched labels: {DATA_ROOT}/cbis_unmatched_labels.csv')

print(f'\nSaved train/test/final CSV files to: {DATA_ROOT}')

Total labels        : 3568
Total exact matched : 3567
Unmatched labels    : 1
Train               : 2863
Test                : 704

Train labels:
label_name
BENIGN       1682
MALIGNANT    1181
Name: count, dtype: int64

Test labels:
label_name
BENIGN       428
MALIGNANT    276
Name: count, dtype: int64

Train by type:
abn_type       label_name
calcification  BENIGN        1001
               MALIGNANT      544
mass           BENIGN         681
               MALIGNANT      637
dtype: int64

Saved unmatched labels: /root/autodl-tmp/CBIS/cbis_unmatched_labels.csv

Saved train/test/final CSV files to: /root/autodl-tmp/CBIS


**`BCF` cell 21** — CELL 21: Clean segmentation dataset and print remaining full/crop/ROI relationships  
<sub>1 output block(s) preserved</sub>


In [6]:
# CELL 21: Clean segmentation dataset and print remaining full/crop/ROI relationships
# This does NOT delete image files. It only creates CSV tables.

import os
import pandas as pd

keep_condition = (
    (dataset_match_table["has_full_mammogram"] == True) &
    (dataset_match_table["has_cropped_image"] == True) &
    (dataset_match_table["has_roi_mask"] == True) &
    (dataset_match_table["cropped_case_id"] == dataset_match_table["roi_case_id"])
)

clean_segmentation_df = dataset_match_table[keep_condition].copy().reset_index(drop=True)
removed_segmentation_df = dataset_match_table[~keep_condition].copy().reset_index(drop=True)

print("Clean segmentation dataset")
print("=" * 70)
print(f"Original rows : {len(dataset_match_table)}")
print(f"Clean rows    : {len(clean_segmentation_df)}")
print(f"Removed rows  : {len(removed_segmentation_df)}")

print("\nClean rows by split:")
print(clean_segmentation_df["split"].value_counts())

print("\nClean rows by abnormality type:")
print(clean_segmentation_df["abn_type"].value_counts())

print("\nClean rows by cancer type:")
print(clean_segmentation_df["label_name"].value_counts())

clean_path = f"{DATA_ROOT}/clean_segmentation_dataset.csv"
removed_path = f"{DATA_ROOT}/removed_segmentation_rows.csv"

clean_segmentation_df.to_csv(clean_path, index=False)
removed_segmentation_df.to_csv(removed_path, index=False)

print("\nSaved clean rows to:")
print(clean_path)

print("\nSaved removed rows to:")
print(removed_path)

remaining_relationships = clean_segmentation_df[
    [
        "patient_id",
        "full_case_id",
        "cropped_case_id",
        "roi_case_id",
        "pathology",
        "label_name",
        "abn_type",
        "split",
        "full_jpeg_path",
        "cropped_jpeg_path",
        "roi_mask_jpeg_path",
    ]
].copy()

print("\nRemaining full mammogram -> cropped image -> ROI mask rows:")
print(remaining_relationships)

roi_per_full_remaining = (
    clean_segmentation_df
    .groupby("full_case_id")
    .agg(
        patient_id=("patient_id", "first"),
        total_rois=("roi_case_id", "nunique"),
        cancer_types=("label_name", lambda x: ", ".join(sorted(set(x)))),
        abnormality_types=("abn_type", lambda x: ", ".join(sorted(set(x)))),
    )
    .reset_index()
    .sort_values("total_rois", ascending=False)
)

print("\nRemaining ROI count per full mammogram:")
print(roi_per_full_remaining)

print("\nFirst 20 clean relationships:")
print(remaining_relationships.head(20).to_string(index=False))

NameError: name 'dataset_match_table' is not defined

**`BCF` cell 23** — CELL 23: Clean segmentation dataset, filter valid full/crop/ROI pairs & save CSVs to GPU  
<sub>1 output block(s) preserved</sub>


In [33]:
# CELL 23: Clean segmentation dataset, filter valid full/crop/ROI pairs & save CSVs to GPU storage
# Only saves output CSV files, no extra heavy logic, minimal prints
import os
import pandas as pd

# Updated AutoDL GPU path instead of Colab Drive
DATA_ROOT = "/root/autodl-tmp/CBIS"
clean_csv_path = os.path.join(DATA_ROOT, "clean_segmentation_dataset.csv")
removed_csv_path = os.path.join(DATA_ROOT, "removed_segmentation_rows.csv")

# Filter valid samples
keep_condition = (
    (dataset_match_table["has_full_mammogram"] == True) &
    (dataset_match_table["has_cropped_image"] == True) &
    (dataset_match_table["has_roi_mask"] == True) &
    (dataset_match_table["cropped_case_id"] == dataset_match_table["roi_case_id"])
)

clean_segmentation_df = dataset_match_table[keep_condition].copy().reset_index(drop=True)
removed_segmentation_df = dataset_match_table[~keep_condition].copy().reset_index(drop=True)

# Save both outputs permanently into GPU folder
clean_segmentation_df.to_csv(clean_csv_path, index=False)
removed_segmentation_df.to_csv(removed_csv_path, index=False)

print(f"Saved clean dataset -> {clean_csv_path}")
print(f"Saved discarded rows log -> {removed_csv_path}")
print(f"Clean valid samples count: {len(clean_segmentation_df)}")

Saved clean dataset -> /root/autodl-tmp/CBIS/clean_segmentation_dataset.csv
Saved discarded rows log -> /root/autodl-tmp/CBIS/removed_segmentation_rows.csv
Clean valid samples count: 3242


**`BCF` cell 25** — Check image sizes and corrupted files in final clean segmentation dataset  
<sub>0 output block(s) preserved</sub>


In [ ]:
#  Check image sizes and corrupted files in final clean segmentation dataset

# import cv2
# import pandas as pd
# from tqdm import tqdm

# # If clean_segmentation_df is not already loaded, load it
# clean_csv_path = f"{DATA_ROOT}/clean_segmentation_dataset.csv"

# if "clean_segmentation_df" not in globals():
#     clean_segmentation_df = pd.read_csv(clean_csv_path)

# def check_images(df, path_col):
#     records = []
#     bad_paths = []

#     for idx, path in tqdm(
#         enumerate(df[path_col]),
#         total=len(df),
#         desc=f"Checking {path_col}"
#     ):
#         img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

#         if img is None:
#             bad_paths.append(path)
#             records.append({
#                 "row_index": idx,
#                 "path_col": path_col,
#                 "path": path,
#                 "readable": False,
#                 "height": None,
#                 "width": None,
#             })
#             continue

#         h, w = img.shape

#         records.append({
#             "row_index": idx,
#             "path_col": path_col,
#             "path": path,
#             "readable": True,
#             "height": h,
#             "width": w,
#         })

#     return pd.DataFrame(records), bad_paths

# # Check all three image types
# full_size_df, bad_full = check_images(clean_segmentation_df, "full_jpeg_path")
# crop_size_df, bad_crop = check_images(clean_segmentation_df, "cropped_jpeg_path")
# mask_size_df, bad_mask = check_images(clean_segmentation_df, "roi_mask_jpeg_path")

# all_size_df = pd.concat(
#     [full_size_df, crop_size_df, mask_size_df],
#     ignore_index=True
# )

# print("Corrupted / unreadable files")
# print("=" * 70)
# print(f"Unreadable full mammograms : {len(bad_full)}")
# print(f"Unreadable cropped images  : {len(bad_crop)}")
# print(f"Unreadable ROI masks       : {len(bad_mask)}")

# print("\nSize summary")
# print("=" * 70)
# summary = (
#     all_size_df[all_size_df["readable"] == True]
#     .groupby("path_col")
#     .agg(
#         count=("path", "count"),
#         min_height=("height", "min"),
#         max_height=("height", "max"),
#         mean_height=("height", "mean"),
#         min_width=("width", "min"),
#         max_width=("width", "max"),
#         mean_width=("width", "mean"),
#     )
# )

# display(summary)

# # Show biggest images
# print("\nLargest full mammograms:")
# display(
#     full_size_df[full_size_df["readable"] == True]
#     .sort_values(["height", "width"], ascending=False)
#     .head(10)
# )

# print("\nLargest cropped images:")
# display(
#     crop_size_df[crop_size_df["readable"] == True]
#     .sort_values(["height", "width"], ascending=False)
#     .head(10)
# )

# print("\nLargest ROI masks:")
# display(
#     mask_size_df[mask_size_df["readable"] == True]
#     .sort_values(["height", "width"], ascending=False)
#     .head(10)
# )

# # Save results
# all_size_df.to_csv(f"{DATA_ROOT}/clean_segmentation_image_size_check.csv", index=False)

# bad_df = all_size_df[all_size_df["readable"] == False].copy()
# bad_df.to_csv(f"{DATA_ROOT}/clean_segmentation_corrupted_files.csv", index=False)

# print("\nSaved:")
# print(f"{DATA_ROOT}/clean_segmentation_image_size_check.csv")
# print(f"{DATA_ROOT}/clean_segmentation_corrupted_files.csv")

## B · Crop rebuild — v3 through the final 512px tissue-framed crops


**`BCF` cell 236** — ══════════════════════════════════════════════════════════════════════  
<sub>2 output block(s) preserved</sub>


In [7]:
# ══════════════════════════════════════════════════════════════════════
# DEFINITIVE REBUILD using dicom_info.csv (authoritative file labels)
#   SeriesDescription tells us exactly what each file is.
#   PatientName links each ROI mask to its full mammogram.
# ══════════════════════════════════════════════════════════════════════
import os, re, hashlib
import numpy as np, pandas as pd, cv2
D="/root/autodl-tmp/CBIS"
OUT=os.path.join(D,"crops_v3"); os.makedirs(OUT,exist_ok=True)
SIZE=256; PAD=0.15

# 1) load dicom_info and map paths onto your disk
di=pd.read_csv(DICOM_CSV)     # <-- put the csv here
def to_local(p):
    p=str(p).replace("CBIS-DDSM/","")                # -> jpeg/<uid>/<file>.jpg
    return os.path.join(D,p)
di["local"]=di["image_path"].apply(to_local)
di["exists"]=di["local"].apply(os.path.exists)
print("dicom_info rows: "+str(len(di))+" | files found on disk: "+str(int(di.exists.sum())))
print(di.SeriesDescription.value_counts().to_string())

# 2) parse PatientName -> abn, split, patient, side, view, lesion#
pat=re.compile(r"^(Mass|Calc)-(Training|Test)_(P_\d+)_(LEFT|RIGHT)_(CC|MLO)(?:_(\d+))?$")
def parse(n):
    m=pat.match(str(n))
    if not m: return pd.Series([None]*6)
    return pd.Series([m.group(1),m.group(2),m.group(3),m.group(4),m.group(5),m.group(6)])
di[["abn","tset","pid","side","view","lesion"]]=di["PatientName"].apply(parse)
di=di[di.pid.notna() & di.exists]

FULL=di[di.SeriesDescription=="full mammogram images"].copy()
MASK=di[di.SeriesDescription=="ROI mask images"].copy()
print("\nparsed: full="+str(len(FULL))+" | masks="+str(len(MASK)))

# key WITHOUT lesion number = the mammogram that contains the lesion
for df in (FULL,MASK):
    df["key"]=df.abn+"_"+df.pid+"_"+df.side+"_"+df.view
full_map=FULL.drop_duplicates("key").set_index("key")["local"].to_dict()
print("unique mammogram keys: "+str(len(full_map)))

# 3) crop each mask together with ITS full mammogram
rows=[]; ok=0; nofull=0; bad=0
for _,r in MASK.iterrows():
    fp=full_map.get(r["key"])
    if fp is None: nofull+=1; continue
    img=cv2.imread(fp,cv2.IMREAD_GRAYSCALE)
    msk=cv2.imread(r["local"],cv2.IMREAD_GRAYSCALE)
    if img is None or msk is None: bad+=1; continue
    if msk.shape!=img.shape:                       # safety: put mask in image space
        msk=cv2.resize(msk,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
    ys,xs=np.where(msk>127)
    if len(xs)<20: bad+=1; continue
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=img.shape
    py,px=int(PAD*(y1-y0+1)),int(PAD*(x1-x0+1))
    y0,y1=max(0,y0-py),min(h,y1+py+1); x0,x1=max(0,x0-px),min(w,x1+px+1)
    ci=cv2.resize(img[y0:y1,x0:x1],(SIZE,SIZE))
    cm=cv2.resize((msk[y0:y1,x0:x1]>127).astype(np.uint8)*255,(SIZE,SIZE),interpolation=cv2.INTER_NEAREST)
    k=hashlib.md5(r["local"].encode()).hexdigest()
    ip=os.path.join(OUT,k+"_img.png"); mp=os.path.join(OUT,k+"_msk.png")
    cv2.imwrite(ip,ci); cv2.imwrite(mp,cm)
    rows.append(dict(patient_id=r["pid"], abn_type=("mass" if r["abn"]=="Mass" else "calcification"),
                     side=r["side"], view=r["view"], lesion=r["lesion"],
                     orig_split=r["tset"], img=ip, msk=mp, source="CBIS"))
    ok+=1
P=pd.DataFrame(rows)
print("\npaired "+str(ok)+" | no matching mammogram "+str(nofull)+" | unreadable/empty "+str(bad))

# 4) VERIFY: is any "img" secretly a mask?
def midtone(im): return float(((im>=25)&(im<=230)).mean())
chk=[midtone(cv2.imread(p,cv2.IMREAD_GRAYSCALE)) for p in P.img.head(400)]
print("img mid-tone: median "+format(np.median(chk),".3f")+
      " | files that look like masks: "+str(int(sum(c<0.05 for c in chk)))+"/400   (must be 0)")

# 5) attach your benign/malignant labels
lab=pd.concat([pd.read_csv(os.path.join(D,f)) for f in
               ["train_grouped.csv","val_grouped.csv","test_grouped.csv"]],ignore_index=True)
lab=lab[["patient_id","abn_type","label_name","label"]].drop_duplicates(subset=["patient_id","abn_type"])
P=P.merge(lab,on=["patient_id","abn_type"],how="left")
print("labelled: "+str(int(P.label.notna().sum()))+"/"+str(len(P)))
P=P[P.label.notna()].reset_index(drop=True)

# 6) patient-grouped split
plab=P.groupby("patient_id")["label"].max()
rng=np.random.RandomState(42); tr,va,te=[],[],[]
for L in sorted(plab.unique()):
    ids=plab[plab==L].index.tolist(); rng.shuffle(ids)
    n=len(ids); a=int(.70*n); b=int(.15*n)
    tr+=ids[:a]; va+=ids[a:a+b]; te+=ids[a+b:]
P["split"]=P.patient_id.map(lambda p:"train" if p in set(tr) else ("val" if p in set(va) else "test"))
for x,y in [("train","test"),("train","val"),("val","test")]:
    o=set(P[P.split==x].patient_id)&set(P[P.split==y].patient_id)
    print(x+"/"+y+" overlap: "+str(len(o))); assert len(o)==0
P.to_csv(os.path.join(D,"cbis_v3.csv"),index=False)
print("\n"+"="*56)
print(str(dict(P.split.value_counts())))
print("types: "+str(dict(P.abn_type.value_counts())))
print("patients: "+str(P.patient_id.nunique())+" | saved cbis_v3.csv")
print("="*56)

dicom_info rows: 10237 | files found on disk: 10237
SeriesDescription
cropped images           3567
ROI mask images          3247
full mammogram images    2857



parsed: full=2857 | masks=3247
unique mammogram keys: 2857

paired 3242 | no matching mammogram 5 | unreadable/empty 0
img mid-tone: median 1.000 | files that look like masks: 0/400   (must be 0)
labelled: 3242/3242
train/test overlap: 0
train/val overlap: 0
val/test overlap: 0

{'train': np.int64(2301), 'test': np.int64(496), 'val': np.int64(445)}
types: {'mass': np.int64(1696), 'calcification': np.int64(1546)}
patients: 1432 | saved cbis_v3.csv


**`BCF` cell 238** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [9]:
# ══════════════════════════════════════════════════════════════════════
# REBUILD v4 — same as v3, but mask = ALL lesions inside the crop window
# ══════════════════════════════════════════════════════════════════════
import os, re, glob, hashlib
import numpy as np, pandas as pd, cv2
D="/root/autodl-tmp/CBIS"
OUT=os.path.join(D,"crops_v4"); os.makedirs(OUT,exist_ok=True)
SIZE=256; PAD=0.15

DICOM_CSV=glob.glob(os.path.join(D,"**","dicom_info.csv"),recursive=True)[0]
di=pd.read_csv(DICOM_CSV)
di["local"]=di["image_path"].apply(lambda p: os.path.join(D,str(p).replace("CBIS-DDSM/","")))
di=di[di["local"].apply(os.path.exists)]

pat=re.compile(r"^(Mass|Calc)-(Training|Test)_(P_\d+)_(LEFT|RIGHT)_(CC|MLO)(?:_(\d+))?$")
def parse(n):
    m=pat.match(str(n))
    return pd.Series([None]*6) if not m else pd.Series([m.group(i) for i in range(1,7)])
di[["abn","tset","pid","side","view","lesion"]]=di["PatientName"].apply(parse)
di=di[di.pid.notna()]

FULL=di[di.SeriesDescription=="full mammogram images"].copy()
MASK=di[di.SeriesDescription=="ROI mask images"].copy()
for df in (FULL,MASK): df["key"]=df.abn+"_"+df.pid+"_"+df.side+"_"+df.view
full_map=FULL.drop_duplicates("key").set_index("key")["local"].to_dict()

# ← UNION: group all mask files by mammogram
masks_by_key = MASK.groupby("key")["local"].apply(list).to_dict()

rows=[]; ok=0; nofull=0; bad=0; changed=0
for _,r in MASK.iterrows():
    fp=full_map.get(r["key"])
    if fp is None: nofull+=1; continue
    img=cv2.imread(fp,cv2.IMREAD_GRAYSCALE)
    own=cv2.imread(r["local"],cv2.IMREAD_GRAYSCALE)
    if img is None or own is None: bad+=1; continue
    if own.shape!=img.shape:
        own=cv2.resize(own,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)

    # ← UNION: OR together every lesion on this mammogram
    union=np.zeros(img.shape,np.uint8)
    for mp_ in masks_by_key[r["key"]]:
        mm=cv2.imread(mp_,cv2.IMREAD_GRAYSCALE)
        if mm is None: continue
        if mm.shape!=img.shape:
            mm=cv2.resize(mm,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
        union|=(mm>127).astype(np.uint8)

    # window still comes from THIS lesion
    ys,xs=np.where(own>127)
    if len(xs)<20: bad+=1; continue
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=img.shape
    py,px=int(PAD*(y1-y0+1)),int(PAD*(x1-x0+1))
    y0,y1=max(0,y0-py),min(h,y1+py+1); x0,x1=max(0,x0-px),min(w,x1+px+1)

    ci=cv2.resize(img[y0:y1,x0:x1],(SIZE,SIZE))
    cm_own  =(own[y0:y1,x0:x1]>127).astype(np.uint8)
    cm_union= union[y0:y1,x0:x1]
    if cm_union.sum() > cm_own.sum()+50: changed+=1     # ← UNION: did a neighbour appear?
    cm=cv2.resize(cm_union*255,(SIZE,SIZE),interpolation=cv2.INTER_NEAREST)

    k=hashlib.md5(r["local"].encode()).hexdigest()
    ip=os.path.join(OUT,k+"_img.png"); mp2=os.path.join(OUT,k+"_msk.png")
    cv2.imwrite(ip,ci); cv2.imwrite(mp2,cm)
    rows.append(dict(patient_id=r["pid"],
                     abn_type=("mass" if r["abn"]=="Mass" else "calcification"),
                     img=ip, msk=mp2, source="CBIS"))
    ok+=1

P=pd.DataFrame(rows)
print("paired "+str(ok)+" | no mammogram "+str(nofull)+" | bad "+str(bad))
print("crops where a NEIGHBOUR lesion was added: "+str(changed)+"   <-- this is what the union fixed")

lab=pd.concat([pd.read_csv(os.path.join(D,f)) for f in
               ["train_grouped.csv","val_grouped.csv","test_grouped.csv"]],ignore_index=True)
lab=lab[["patient_id","abn_type","label_name","label"]].drop_duplicates(subset=["patient_id","abn_type"])
P=P.merge(lab,on=["patient_id","abn_type"],how="left")
P=P[P.label.notna()].reset_index(drop=True)

plab=P.groupby("patient_id")["label"].max()
rng=np.random.RandomState(42); tr,va,te=[],[],[]
for L in sorted(plab.unique()):
    ids=plab[plab==L].index.tolist(); rng.shuffle(ids)
    n=len(ids); a=int(.70*n); b=int(.15*n)
    tr+=ids[:a]; va+=ids[a:a+b]; te+=ids[a+b:]
P["split"]=P.patient_id.map(lambda p:"train" if p in set(tr) else ("val" if p in set(va) else "test"))
for x,y in [("train","test"),("train","val"),("val","test")]:
    o=set(P[P.split==x].patient_id)&set(P[P.split==y].patient_id)
    assert len(o)==0, x+"/"+y+" leak"
P.to_csv(os.path.join(D,"cbis_v4.csv"),index=False)
print("\n"+str(dict(P.split.value_counts()))+" | patients "+str(P.patient_id.nunique()))
print("saved cbis_v4.csv")

paired 3242 | no mammogram 5 | bad 0
crops where a NEIGHBOUR lesion was added: 93   <-- this is what the union fixed

{'train': np.int64(2301), 'test': np.int64(496), 'val': np.int64(445)} | patients 1432
saved cbis_v4.csv


**`BCF` cell 242** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════
# ATTACH official split + morphology features to cbis_v4.csv
# ══════════════════════════════════════════════════════════════════════
import os, re, glob, hashlib
import numpy as np, pandas as pd
D="/root/autodl-tmp/CBIS"

# --- rebuild the mask -> (patient, side, view, lesion, official split) map
DICOM=glob.glob(os.path.join(D,"**","dicom_info.csv"),recursive=True)[0]
di=pd.read_csv(DICOM)
di["local"]=di["image_path"].apply(lambda p: os.path.join(D,str(p).replace("CBIS-DDSM/","")))
pat=re.compile(r"^(Mass|Calc)-(Training|Test)_(P_\d+)_(LEFT|RIGHT)_(CC|MLO)(?:_(\d+))?$")
def parse(n):
    m=pat.match(str(n))
    return pd.Series([None]*6) if not m else pd.Series([m.group(i) for i in range(1,7)])
di[["abn","tset","pid","side","view","lesion"]]=di["PatientName"].apply(parse)
MASK=di[(di.SeriesDescription=="ROI mask images") & di.pid.notna()].copy()
MASK["md5"]=MASK["local"].apply(lambda p: hashlib.md5(p.encode()).hexdigest())
MASK["official_split"]=MASK["tset"].str.lower().replace({"training":"train"})

# --- join back onto cbis_v4 via the md5 in the mask filename
P=pd.read_csv(os.path.join(D,"cbis_v4.csv"))
P["md5"]=P["msk"].apply(lambda p: os.path.basename(p).replace("_msk.png",""))
P=P.merge(MASK[["md5","side","view","lesion","official_split"]],on="md5",how="left")
print("rows with official split recovered: "+str(int(P.official_split.notna().sum()))+"/"+str(len(P)))
print("official split counts: "+str(dict(P.official_split.value_counts())))
print("  by type:")
print(P.groupby(["abn_type","official_split"]).size().to_string())

# --- attach morphology from the description CSVs
frames=[]
for f in glob.glob(os.path.join(D,"*case_description*set.csv")):
    d=pd.read_csv(f); d["__src"]=os.path.basename(f); frames.append(d)
if not frames:
    print("\n!! description CSVs not found in "+D+" — upload them there")
else:
    DESC=pd.concat(frames,ignore_index=True)
    DESC.columns=[c.strip() for c in DESC.columns]
    DESC=DESC.rename(columns={
        "left or right breast":"side","image view":"view",
        "abnormality id":"lesion","abnormality type":"abn_desc",
        "breast density":"density","calc type":"calc_type",
        "calc distribution":"calc_dist","mass shape":"mass_shape",
        "mass margins":"mass_margins"})
    DESC["lesion"]=DESC["lesion"].astype(str)
    P["lesion"]=P["lesion"].astype(str)
    keep=[c for c in ["patient_id","side","view","lesion","density","assessment",
                      "subtlety","pathology","calc_type","calc_dist",
                      "mass_shape","mass_margins"] if c in DESC.columns]
    DESC=DESC[keep].drop_duplicates(subset=["patient_id","side","view","lesion"])
    P=P.merge(DESC,on=["patient_id","side","view","lesion"],how="left")
    print("\nmorphology attached: "+str(int(P.pathology.notna().sum()))+"/"+str(len(P)))
    print("pathology: "+str(dict(P.pathology.value_counts(dropna=False))))
    for c in ["subtlety","density","assessment"]:
        if c in P: print("  "+c+": "+str(dict(P[c].value_counts(dropna=False).sort_index())))

    # sanity: does CBIS pathology agree with your existing label?
    if "pathology" in P and "label" in P:
        m=P.dropna(subset=["pathology"]).copy()
        m["cbis_mal"]=(m.pathology=="MALIGNANT").astype(int)
        agree=(m.cbis_mal==m.label).mean()
        print("\nlabel agreement with CBIS pathology: "+format(agree,".4f")+
              "   (BENIGN_WITHOUT_CALLBACK treated as benign)")
        if agree<0.99: print("  !! labels disagree — check how you mapped BENIGN_WITHOUT_CALLBACK")

P.to_csv(os.path.join(D,"cbis_v5.csv"),index=False)
print("\nsaved cbis_v5.csv  (adds side, view, lesion, official_split, morphology)")

# --- the count check
n_calc=int((P.abn_type=="calcification").sum())
print("\ncalc lesions in your data: "+str(n_calc))
print("official calc train=1546, test=326, total=1872")
if n_calc==1546:
    print("  !! WARNING: exactly the TRAIN count. The 326 official test calc lesions")
    print("     may be missing. Check official_split counts above.")

rows with official split recovered: 3242/3242
official split counts: {'train': np.int64(2863), 'test': np.int64(379)}
  by type:
abn_type       official_split
calcification  test                 1
               train             1545
mass           test               378
               train             1318

!! description CSVs not found in /root/autodl-tmp/CBIS — upload them there

saved cbis_v5.csv  (adds side, view, lesion, official_split, morphology)

calc lesions in your data: 1546
official calc train=1546, test=326, total=1872
  !! WARNING: exactly the TRAIN count. The 326 official test calc lesions
     may be missing. Check official_split counts above.


**`BCF` cell 243** — import pandas as pd, glob, os, re  
<sub>1 output block(s) preserved</sub>


In [2]:
import pandas as pd, glob, os, re
D="/root/autodl-tmp/CBIS"
di=pd.read_csv(glob.glob(os.path.join(D,"**","dicom_info.csv"),recursive=True)[0])
m=di[di.SeriesDescription=="ROI mask images"].copy()
m["grp"]=m.PatientName.astype(str).str.extract(r"^(Mass|Calc)-(Training|Test)")[0]+"-"+ \
         m.PatientName.astype(str).str.extract(r"^(Mass|Calc)-(Training|Test)")[1]
print(m.grp.value_counts().to_string())
print("\nExpected: Calc-Training 1546 | Calc-Test 326 | Mass-Training 1318 | Mass-Test 378")

grp
Calc-Training    1545
Mass-Training    1318
Mass-Test         378
Calc-Test           6

Expected: Calc-Training 1546 | Calc-Test 326 | Mass-Training 1318 | Mass-Test 378


**`BCF` cell 245** — import pandas as pd, glob, os, cv2, numpy as np  
<sub>1 output block(s) preserved</sub>


In [5]:
import pandas as pd, glob, os, cv2, numpy as np
D="/root/autodl-tmp/CBIS"
di=pd.read_csv(glob.glob(os.path.join(D,"**","dicom_info.csv"),recursive=True)[0])

ct = di[di.PatientName.astype(str).str.startswith("Calc-Test")]
print("Calc-Test rows in dicom_info: "+str(len(ct)))
print(ct.SeriesDescription.value_counts(dropna=False).to_string())
print()
print("rows with NO SeriesDescription (whole file): "+str(int(di.SeriesDescription.isna().sum())))
print()

# look inside ONE Calc-Test series folder: what's actually there?
sub=ct[ct.SeriesDescription=="cropped images"].head(1)
if len(sub):
    uid=sub.iloc[0]["SeriesInstanceUID"]
    same=di[di.SeriesInstanceUID==uid]
    print("files in that series ("+str(len(same))+"):")
    for _,r in same.iterrows():
        p=os.path.join(D,str(r["image_path"]).replace("CBIS-DDSM/",""))
        im=cv2.imread(p,cv2.IMREAD_GRAYSCALE)
        if im is None: print("   MISSING "+p); continue
        uq=len(np.unique(im[::8,::8]))
        mid=float(((im>=25)&(im<=230)).mean())
        kind="MASK" if mid<0.05 else "image"
        print("   "+os.path.basename(p).ljust(12)+" shape="+str(im.shape).ljust(14)+
              " uniq="+str(uq).ljust(5)+" midtone="+format(mid,".3f")+"  -> "+kind)

Calc-Test rows in dicom_info: 370
SeriesDescription
cropped images           326
full mammogram images     38
ROI mask images            6

rows with NO SeriesDescription (whole file): 566

files in that series (2):
   1-052.jpg    shape=(97, 97)       uniq=97    midtone=0.975  -> image
   2-204.jpg    shape=(4560, 3104)   uniq=3     midtone=0.000  -> MASK


**`BCF` cell 246** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [6]:
# ══════════════════════════════════════════════════════════════════════
# v5 REBUILD — find masks by PIXEL CONTENT, not by SeriesDescription
#   Recovers the ~320 Calc-Test masks hiding in "cropped images" series
# ══════════════════════════════════════════════════════════════════════
import os, re, glob, hashlib
import numpy as np, pandas as pd, cv2
D="/root/autodl-tmp/CBIS"
OUT=os.path.join(D,"crops_v5"); os.makedirs(OUT,exist_ok=True)
SIZE=256; PAD=0.15

di=pd.read_csv(glob.glob(os.path.join(D,"**","dicom_info.csv"),recursive=True)[0])
di["local"]=di["image_path"].apply(lambda p: os.path.join(D,str(p).replace("CBIS-DDSM/","")))
di=di[di["local"].apply(os.path.exists)]
pat=re.compile(r"^(Mass|Calc)-(Training|Test)_(P_\d+)_(LEFT|RIGHT)_(CC|MLO)(?:_(\d+))?$")
def parse(n):
    m=pat.match(str(n))
    return pd.Series([None]*6) if not m else pd.Series([m.group(i) for i in range(1,7)])
di[["abn","tset","pid","side","view","lesion"]]=di["PatientName"].apply(parse)
di=di[di.pid.notna()].copy()

def midtone(im): return float(((im>=25)&(im<=230)).mean())

# classify EVERY file by what it actually contains
kinds=[]
for p in di["local"]:
    im=cv2.imread(p,cv2.IMREAD_GRAYSCALE)
    if im is None: kinds.append((None,None)); continue
    kinds.append(("mask" if midtone(im)<0.05 else "image", im.shape))
di["kind"]=[k[0] for k in kinds]
di["shape"]=[k[1] for k in kinds]
di=di[di.kind.notna()]
print("by ACTUAL content: "+str(dict(di.kind.value_counts())))

MASKS=di[di.kind=="mask"].copy()
IMGS =di[di.kind=="image"].copy()
MASKS["grp"]=MASKS.abn+"-"+MASKS.tset
print("\nmasks found per group:")
print(MASKS.grp.value_counts().to_string())
print("expected: Calc-Training 1546 | Calc-Test 326 | Mass-Training 1318 | Mass-Test 378")

# full mammogram = grayscale file whose shape matches the mask
IMGS["key"]=IMGS.abn+"_"+IMGS.pid+"_"+IMGS.side+"_"+IMGS.view
MASKS["key"]=MASKS.abn+"_"+MASKS.pid+"_"+MASKS.side+"_"+MASKS.view
by_key_shape={}
for _,r in IMGS.iterrows():
    by_key_shape.setdefault((r["key"],r["shape"]),r["local"])

masks_by_key=MASKS.groupby("key")["local"].apply(list).to_dict()

rows=[]; ok=0; nofull=0
for _,r in MASKS.iterrows():
    fp=by_key_shape.get((r["key"],r["shape"]))
    if fp is None: nofull+=1; continue
    img=cv2.imread(fp,cv2.IMREAD_GRAYSCALE)
    own=cv2.imread(r["local"],cv2.IMREAD_GRAYSCALE)
    if img is None or own is None: nofull+=1; continue
    union=np.zeros(img.shape,np.uint8)                    # all lesions on this mammogram
    for mp_ in masks_by_key[r["key"]]:
        mm=cv2.imread(mp_,cv2.IMREAD_GRAYSCALE)
        if mm is None or mm.shape!=img.shape: continue
        union|=(mm>127).astype(np.uint8)
    ys,xs=np.where(own>127)
    if len(xs)<20: nofull+=1; continue
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=img.shape
    py,px=int(PAD*(y1-y0+1)),int(PAD*(x1-x0+1))
    y0,y1=max(0,y0-py),min(h,y1+py+1); x0,x1=max(0,x0-px),min(w,x1+px+1)
    ci=cv2.resize(img[y0:y1,x0:x1],(SIZE,SIZE))
    cm=cv2.resize(union[y0:y1,x0:x1]*255,(SIZE,SIZE),interpolation=cv2.INTER_NEAREST)
    k=hashlib.md5(r["local"].encode()).hexdigest()
    ip=os.path.join(OUT,k+"_img.png"); mp2=os.path.join(OUT,k+"_msk.png")
    cv2.imwrite(ip,ci); cv2.imwrite(mp2,cm)
    rows.append(dict(patient_id=r["pid"],
                     abn_type=("mass" if r["abn"]=="Mass" else "calcification"),
                     side=r["side"], view=r["view"], lesion=r["lesion"],
                     official_split=("train" if r["tset"]=="Training" else "test"),
                     img=ip, msk=mp2, source="CBIS"))
    ok+=1

P=pd.DataFrame(rows)
print("\npaired "+str(ok)+"  | no full mammogram "+str(nofull))
print(P.groupby(["abn_type","official_split"]).size().to_string())

# attach morphology (recursive glob - your CSVs are in CBIS/csv/)
frames=[pd.read_csv(f) for f in glob.glob(os.path.join(D,"**","*case_description*set.csv"),recursive=True)]
DESC=pd.concat(frames,ignore_index=True)
DESC=DESC.rename(columns={"left or right breast":"side","image view":"view",
                          "abnormality id":"lesion","breast density":"density",
                          "breast_density":"density","calc type":"calc_type",
                          "calc distribution":"calc_dist","mass shape":"mass_shape",
                          "mass margins":"mass_margins"})
DESC["lesion"]=DESC["lesion"].astype(str); P["lesion"]=P["lesion"].astype(str)
keep=[c for c in ["patient_id","side","view","lesion","density","assessment","subtlety",
                  "pathology","calc_type","calc_dist","mass_shape","mass_margins"] if c in DESC.columns]
DESC=DESC[keep].drop_duplicates(subset=["patient_id","side","view","lesion"])
P=P.merge(DESC,on=["patient_id","side","view","lesion"],how="left")
print("\nmorphology attached: "+str(int(P.pathology.notna().sum()))+"/"+str(len(P)))

# label: BENIGN_WITHOUT_CALLBACK -> benign (state this in your methods)
P["label"]=(P.pathology=="MALIGNANT").astype(int)
P["label_name"]=P.label.map({1:"MALIGNANT",0:"BENIGN"})
P=P[P.pathology.notna()].reset_index(drop=True)

ov=set(P[P.official_split=="train"].patient_id)&set(P[P.official_split=="test"].patient_id)
print("official train/test patient overlap: "+str(len(ov))+"  (must be 0)")
P.to_csv(os.path.join(D,"cbis_v5.csv"),index=False)
print("\nsaved cbis_v5.csv | "+str(len(P))+" lesions | "+str(P.patient_id.nunique())+" patients")
print(str(dict(P.official_split.value_counts())))

by ACTUAL content: {'image': np.int64(6423), 'mask': np.int64(3248)}

masks found per group:
grp
Calc-Training    1546
Mass-Training    1318
Mass-Test         378
Calc-Test           6
expected: Calc-Training 1546 | Calc-Test 326 | Mass-Training 1318 | Mass-Test 378

paired 3164  | no full mammogram 84
abn_type       official_split
calcification  test                 1
               train             1545
mass           test               365
               train             1253

morphology attached: 3164/3164
official train/test patient overlap: 18  (must be 0)

saved cbis_v5.csv | 3164 lesions | 1393 patients
{'train': np.int64(2798), 'test': np.int64(366)}


**`BCF` cell 247** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [7]:
# ══════════════════════════════════════════════════════════════════════
# v6 REBUILD — description CSVs are the index (authoritative, no metadata)
# ══════════════════════════════════════════════════════════════════════
import os, glob, hashlib
import numpy as np, pandas as pd, cv2
D="/root/autodl-tmp/CBIS"; JPEG=os.path.join(D,"jpeg")
OUT=os.path.join(D,"crops_v6"); os.makedirs(OUT,exist_ok=True)
SIZE=256; PAD=0.15

def series_of(p):
    """extract the SeriesInstanceUID (folder name) from a CBIS file path"""
    if not isinstance(p,str): return None
    parts=[x for x in p.strip().replace("\\","/").split("/") if x]
    return parts[2] if len(parts)>=3 else None

def midtone(im): return float(((im>=25)&(im<=230)).mean())

def files_in(series):
    if not series: return []
    return sorted(glob.glob(os.path.join(JPEG,series,"*.jpg")))

def pick(series, want):
    """want='mask' -> the binary file; want='image' -> the grayscale file"""
    best=None; best_px=-1
    for f in files_in(series):
        im=cv2.imread(f,cv2.IMREAD_GRAYSCALE)
        if im is None: continue
        is_mask = midtone(im)<0.05
        if (want=="mask") != is_mask: continue
        px=im.shape[0]*im.shape[1]
        if px>best_px: best,best_px=f,px      # take the largest match
    return best

# ---- build the lesion index from the 4 description CSVs ----
frames=[]
for f in glob.glob(os.path.join(D,"**","*case_description*set.csv"),recursive=True):
    d=pd.read_csv(f)
    d.columns=[c.strip() for c in d.columns]
    d["official_split"]="test" if "test" in os.path.basename(f).lower() else "train"
    frames.append(d)
DESC=pd.concat(frames,ignore_index=True)
DESC=DESC.rename(columns={"left or right breast":"side","image view":"view",
                          "abnormality id":"lesion","abnormality type":"abn_type",
                          "breast density":"density","breast_density":"density",
                          "calc type":"calc_type","calc distribution":"calc_dist",
                          "mass shape":"mass_shape","mass margins":"mass_margins",
                          "image file path":"full_path","ROI mask file path":"mask_path"})
print("lesions in description CSVs: "+str(len(DESC)))
print(DESC.groupby(["abn_type","official_split"]).size().to_string())

DESC["mask_series"]=DESC["mask_path"].apply(series_of)
DESC["full_series"]=DESC["full_path"].apply(series_of)

rows=[]; ok=0; nomask=0; nofull=0
for _,r in DESC.iterrows():
    mf=pick(r["mask_series"],"mask")
    ff=pick(r["full_series"],"image")
    if mf is None: nomask+=1; continue
    if ff is None: nofull+=1; continue
    mask=cv2.imread(mf,cv2.IMREAD_GRAYSCALE)
    img =cv2.imread(ff,cv2.IMREAD_GRAYSCALE)
    if mask is None or img is None: nomask+=1; continue
    if mask.shape!=img.shape:
        mask=cv2.resize(mask,(img.shape[1],img.shape[0]),interpolation=cv2.INTER_NEAREST)
    ys,xs=np.where(mask>127)
    if len(xs)<20: nomask+=1; continue
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max(); h,w=img.shape
    py,px=int(PAD*(y1-y0+1)),int(PAD*(x1-x0+1))
    y0,y1=max(0,y0-py),min(h,y1+py+1); x0,x1=max(0,x0-px),min(w,x1+px+1)
    ci=cv2.resize(img[y0:y1,x0:x1],(SIZE,SIZE))
    cm=cv2.resize((mask[y0:y1,x0:x1]>127).astype(np.uint8)*255,(SIZE,SIZE),interpolation=cv2.INTER_NEAREST)
    k=hashlib.md5(mf.encode()).hexdigest()
    ip=os.path.join(OUT,k+"_img.png"); mp=os.path.join(OUT,k+"_msk.png")
    cv2.imwrite(ip,ci); cv2.imwrite(mp,cm)
    d=r.to_dict()
    d.update(img=ip, msk=mp, source="CBIS",
             abn_type=("mass" if "mass" in str(r["abn_type"]).lower() else "calcification"),
             label=int(str(r["pathology"])=="MALIGNANT"))
    rows.append(d); ok+=1

P=pd.DataFrame(rows)
print("\npaired "+str(ok)+" | mask not found "+str(nomask)+" | full mammogram not found "+str(nofull))
print(P.groupby(["abn_type","official_split"]).size().to_string())
print("\nEXPECTED: calc train 1546 / test 326 | mass train 1318 / test 378")

# CBIS quirk: a patient can be in mass-train AND calc-test.
mtr=set(P[(P.abn_type=="mass")&(P.official_split=="train")].patient_id)
cte=set(P[(P.abn_type=="calcification")&(P.official_split=="test")].patient_id)
ctr=set(P[(P.abn_type=="calcification")&(P.official_split=="train")].patient_id)
mte=set(P[(P.abn_type=="mass")&(P.official_split=="test")].patient_id)
print("\nwithin-mass  train/test overlap: "+str(len(mtr&mte))+"   (must be 0)")
print("within-calc  train/test overlap: "+str(len(ctr&cte))+"   (must be 0)")
print("cross-type overlap (mass-train ∩ calc-test): "+str(len(mtr&cte)))
print("  -> this is a known CBIS quirk. Train MASS and CALC models SEPARATELY,")
print("     each on its own official split. No leakage that way.")

P.to_csv(os.path.join(D,"cbis_v6.csv"),index=False)
print("\nsaved cbis_v6.csv | "+str(len(P))+" lesions | "+str(P.patient_id.nunique())+" patients")

lesions in description CSVs: 3568
abn_type       official_split
calcification  test               326
               train             1546
mass           test               378
               train             1318

paired 3566 | mask not found 2 | full mammogram not found 0
abn_type       official_split
calcification  test               326
               train             1544
mass           test               378
               train             1318

EXPECTED: calc train 1546 / test 326 | mass train 1318 / test 378

within-mass  train/test overlap: 0   (must be 0)
within-calc  train/test overlap: 0   (must be 0)
cross-type overlap (mass-train ∩ calc-test): 13
  -> this is a known CBIS quirk. Train MASS and CALC models SEPARATELY,
     each on its own official split. No leakage that way.

saved cbis_v6.csv | 3566 lesions | 1566 patients


**`BCF` cell 249** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [9]:
# ══════════════════════════════════════════════════════════════════════
# ALIGNMENT VERIFICATION — every mass + calc pair in cbis_v6.csv
#   For each lesion, checks:
#     - image is a real mammogram (grey tones), not a mask
#     - mask is binary, non-empty, not covering the whole crop
#     - mask sits on TISSUE, not on black background   <- alignment proof
#     - GT mask centroid vs. brightest region of the image
#   Writes align_report.csv + contact sheets per group
# ══════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"
FIG=os.path.join(D,"figures","align_v6"); os.makedirs(FIG,exist_ok=True)
P=pd.read_csv(os.path.join(D,"cbis_v6.csv"))
print("checking "+str(len(P))+" lesions\n")

rows=[]
for i,r in P.iterrows():
    img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
    msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    if img is None or msk is None:
        rows.append(dict(i=i,ok=False,why="unreadable")); continue
    m=(msk>127).astype(np.uint8)
    midtone=float(((img>=25)&(img<=230)).mean())      # is the IMAGE a real mammogram?
    area=float(m.mean())
    if m.sum()<10:
        rows.append(dict(i=i,ok=False,why="empty mask",area=area,midtone=midtone)); continue

    inside = img[m>0]
    outside= img[m==0]
    black_in_mask = float((inside<15).mean())         # mask sitting on black padding = MISALIGNED
    mean_in  = float(inside.mean())
    mean_out = float(outside.mean()) if outside.size else 0.0
    contrast = mean_in-mean_out                       # lesion should be BRIGHTER than surround

    # centroid of mask vs centroid of the brightest 2% of pixels
    ys,xs=np.nonzero(m); cy,cx=ys.mean(),xs.mean()
    thr=np.percentile(img,98)
    by,bx=np.nonzero(img>=thr)
    dist = float(np.hypot(cx-bx.mean(), cy-by.mean())) if len(bx) else np.nan

    flags=[]
    if midtone<0.10:          flags.append("IMG_IS_MASK")
    if black_in_mask>0.50:    flags.append("MASK_ON_BLACK")
    if area>0.90:             flags.append("MASK_COVERS_ALL")
    if area<0.01:             flags.append("MASK_TINY")
    rows.append(dict(i=i, abn=r["abn_type"], split=r["official_split"],
                     ok=(len(flags)==0), why=",".join(flags) or "ok",
                     area=area, midtone=midtone, black_in_mask=black_in_mask,
                     mean_in=mean_in, mean_out=mean_out, contrast=contrast,
                     centroid_dist=dist))
A=pd.DataFrame(rows)
A.to_csv(os.path.join(D,"align_report.csv"),index=False)

print("="*70)
print("ALIGNMENT REPORT")
print("="*70)
print("  PASS: "+str(int(A.ok.sum()))+"/"+str(len(A))+
      "   FAIL: "+str(int((~A.ok).sum())))
bad=A[~A.ok]
if len(bad):
    print("\n  failure reasons:")
    print(bad.why.value_counts().to_string())
print("\n  " + "-"*66)
for (t,s),sub in A[A.ok].groupby(["abn","split"]):
    print("  "+str(t).ljust(14)+str(s).ljust(7)+"n="+str(len(sub)).rjust(5)+
          " | mask area "+format(sub.area.median(),".3f")+
          " | mask-vs-bg contrast "+format(sub.contrast.median(),"+.1f")+
          " | black-in-mask "+format(sub.black_in_mask.median(),".3f"))
print("\n  READ IT:")
print("   mask area   ~0.2-0.5  = healthy lesion crop")
print("   contrast    POSITIVE  = mask sits on brighter tissue (lesion), good")
print("   contrast    NEGATIVE  = mask sits on DARKER tissue -> suspicious")
print("   black_in_mask near 0  = mask is on tissue, not on padding -> ALIGNED")
print("="*70)

# ---------- contact sheets ----------
def sheet(sub,name,n=8):
    sub=sub.reset_index(drop=True)
    if len(sub)==0: return
    N=min(n,len(sub))
    fig,ax=plt.subplots(N,4,figsize=(12,2.9*N))
    if N==1: ax=ax.reshape(1,4)
    for k in range(N):
        rr=sub.iloc[k*max(1,len(sub)//N)]
        r=P.iloc[int(rr["i"])]
        img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
        msk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
        m=(msk>127).astype(np.uint8)
        cnt,_=cv2.findContours(m,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        out=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        cv2.drawContours(out,cnt,-1,(255,60,50),2)          # red outline only
        ov=cv2.cvtColor(img,cv2.COLOR_GRAY2RGB)
        ov[m>0]=(0.55*ov[m>0]+np.array([0,150,0])).astype(np.uint8)
        for c,(im,t,cm) in enumerate([
                (img,"image","gray"),
                (m*255,"mask  area="+format(rr["area"],".3f"),"gray"),
                (out,"outline (red)",None),
                (ov,"filled  contrast="+format(rr["contrast"],"+.0f"),None)]):
            ax[k,c].imshow(im,cmap=cm) if cm else ax[k,c].imshow(im)
            ax[k,c].set_title(t,fontsize=7); ax[k,c].axis("off")
    plt.suptitle(name,fontsize=12); plt.tight_layout()
    o=os.path.join(FIG,name.lower().replace(" ","_").replace("-","_")+".png")
    plt.savefig(o,dpi=125,bbox_inches="tight"); plt.close(); print("  saved "+o)

print("\ncontact sheets:")
for t in ["mass","calcification"]:
    for s in ["train","test"]:
        sheet(A[(A.abn==t)&(A.split==s)&(A.ok)], t.upper()+"-"+s.upper())
if len(bad): sheet(bad,"FAILED CASES",n=10)

checking 3566 lesions

ALIGNMENT REPORT
  PASS: 3562/3566   FAIL: 4

  failure reasons:
why
IMG_IS_MASK,MASK_ON_BLACK    4

  ------------------------------------------------------------------
  calcification test   n=  326 | mask area 0.445 | mask-vs-bg contrast +7.9 | black-in-mask 0.000
  calcification train  n= 1540 | mask area 0.444 | mask-vs-bg contrast +7.3 | black-in-mask 0.000
  mass          test   n=  378 | mask area 0.378 | mask-vs-bg contrast +20.1 | black-in-mask 0.000
  mass          train  n= 1318 | mask area 0.380 | mask-vs-bg contrast +21.1 | black-in-mask 0.000

  READ IT:
   mask area   ~0.2-0.5  = healthy lesion crop
   contrast    POSITIVE  = mask sits on brighter tissue (lesion), good
   contrast    NEGATIVE  = mask sits on DARKER tissue -> suspicious
   black_in_mask near 0  = mask is on tissue, not on padding -> ALIGNED

contact sheets:
  saved /root/autodl-tmp/CBIS/figures/align_v6/mass_train.png
  saved /root/autodl-tmp/CBIS/figures/align_v6/mass_test.png
  s

**`BCF` cell 332** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════
# STEP 1: does MASS have the same two-file / whole-breast mask bug as calc?
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd, cv2
D="/root/autodl-tmp/CBIS"; JP=os.path.join(D,"jpeg")
raw=pd.read_csv(os.path.join(D,"cbis_final.csv"))
mass=raw[raw.abn_type=="mass"].reset_index(drop=True)
print("mass rows in cbis_final:", len(mass))

def series_uid(p):
    u=[x for x in str(p).split("/") if x.startswith("1.3.6")]; return u[-1] if u else None
def folder(uid):
    f=os.path.join(JP,uid) if uid else None
    return f if f and os.path.isdir(f) else None

# 1. check how many mask folders have TWO files (the bug signature)
two_file=0; one_file=0; checked=0
frac_cover=[]
for _,r in mass.sample(min(150,len(mass)),random_state=0).iterrows():
    f=folder(series_uid(r["mask_path"]))
    if not f: continue
    files=[x for x in os.listdir(f) if x.lower().endswith((".jpg",".jpeg",".png"))]
    checked+=1
    if len(files)>=2: two_file+=1
    else: one_file+=1
    # measure coverage of the file the pipeline would grab
    # check the LARGEST file (full-res) coverage
    best=None; bestsz=0
    for fn in files:
        im=cv2.imread(os.path.join(f,fn),cv2.IMREAD_GRAYSCALE)
        if im is not None and im.size>bestsz: bestsz=im.size; best=im
    if best is not None:
        frac_cover.append((best>60).mean())

print("\nmask folders checked:", checked)
print("  with TWO files (bug signature):", two_file, "("+format(100*two_file/max(checked,1),".0f")+"%)")
print("  with ONE file:", one_file)
print()
frac_cover=np.array(frac_cover)
print("full-size mask coverage of image: median "+format(100*np.median(frac_cover),".0f")+"%")
print("  masks covering >80% (broken):", format(100*(frac_cover>0.8).mean(),".0f")+"%")
print()

# 2. check the CURRENT mass crops if they exist
for cand in ["cbis_mass_fixed.csv","crops_fixed_mass"]:
    p=os.path.join(D,cand)
    print(cand, "exists:", os.path.exists(p))

# 3. what does the existing mass classification data use?
if "abn_type" in raw.columns:
    mm=raw[raw.abn_type=="mass"]
    if "img" in mm.columns:
        folders=mm["img"].astype(str).str.split("/").str[-2].value_counts()
        print("\ncurrent mass crops point to:", dict(folders.head(3)))
        # check a few mask coverages
        cov=[]
        for _,r in mm.dropna(subset=["msk"]).sample(min(60,len(mm)),random_state=0).iterrows():
            mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
            if mk is not None: cov.append((mk>127).mean())
        if cov:
            print("current mass crop mask coverage: median "+format(100*np.median(cov),".0f")+"%  (want ~15-40%, NOT ~88%)")

mass rows in cbis_final: 1696

mask folders checked: 150
  with TWO files (bug signature): 144 (96%)
  with ONE file: 6

full-size mask coverage of image: median 0%
  masks covering >80% (broken): 0%

cbis_mass_fixed.csv exists: False
crops_fixed_mass exists: False

current mass crops point to: {'crops_v6': np.int64(1696)}
current mass crop mask coverage: median 36%  (want ~15-40%, NOT ~88%)


**`BCF` cell 333** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════
# REGENERATE MASS CROPS — same clean pipeline as fixed calcification
#   full-size mask selection, tissue-framed, 512px
#   writes crops_fixed_mass/ + cbis_mass_fixed.csv
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd, cv2
D="/root/autodl-tmp/CBIS"; JP=os.path.join(D,"jpeg")
OUT=os.path.join(D,"crops_fixed_mass"); os.makedirs(OUT,exist_ok=True)
raw=pd.read_csv(os.path.join(D,"cbis_final.csv"))
mass=raw[raw.abn_type=="mass"].reset_index(drop=True)
TARGET=512; PAD=0.30

def series_uid(p):
    u=[x for x in str(p).split("/") if x.startswith("1.3.6")]; return u[-1] if u else None
def pick_fullsize(uid):
    f=os.path.join(JP,uid) if uid else None
    if not f or not os.path.isdir(f): return None
    best=None; bestsz=0
    for fn in os.listdir(f):
        im=cv2.imread(os.path.join(f,fn),cv2.IMREAD_GRAYSCALE)
        if im is not None and im.size>bestsz: bestsz=im.size; best=os.path.join(f,fn)
    return best
def breast_mask(full):
    g=cv2.GaussianBlur(full,(9,9),0)
    _,bw=cv2.threshold(g,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    bw=(bw>0).astype(np.uint8)
    if bw.mean()<0.05: bw=(full>10).astype(np.uint8)
    n,lab,st,_=cv2.connectedComponentsWithStats(bw,8)
    if n>1:
        big=1+int(np.argmax(st[1:,cv2.CC_STAT_AREA])); bw=(lab==big).astype(np.uint8)
    return bw

done=0; skip=0; rows=[]; cov=[]
for _,r in mass.iterrows():
    fp=pick_fullsize(series_uid(r["full_path"])); mp=pick_fullsize(series_uid(r["mask_path"]))
    if not fp or not mp: skip+=1; continue
    full=cv2.imread(fp,cv2.IMREAD_GRAYSCALE); mask=cv2.imread(mp,cv2.IMREAD_GRAYSCALE)
    if full is None or mask is None: skip+=1; continue
    if mask.shape!=full.shape:
        mask=cv2.resize(mask,(full.shape[1],full.shape[0]),interpolation=cv2.INTER_NEAREST)
    if (mask>60).mean()>0.5: mask=255-mask       # invert if mostly white
    br=breast_mask(full)
    lesion=((mask>60)&(br>0)).astype(np.uint8)
    ys,xs=np.where(lesion>0)
    if len(ys)<10:
        ys,xs=np.where(mask>60)
    if len(ys)<10: skip+=1; continue
    frac=(mask>60).mean(); cov.append(frac)
    if frac>0.6: skip+=1; continue
    y0,y1,x0,x1=ys.min(),ys.max(),xs.min(),xs.max()
    h=y1-y0; w=x1-x0; side=max(int(max(h,w)*(1+2*PAD)),64)
    cy,cx=(y0+y1)//2,(x0+x1)//2
    Y0=max(0,cy-side//2); X0=max(0,cx-side//2)
    Y1=min(full.shape[0],Y0+side); X1=min(full.shape[1],X0+side)
    crop=full[Y0:Y1,X0:X1]; cm=mask[Y0:Y1,X0:X1]
    if crop.size==0: skip+=1; continue
    interp=cv2.INTER_AREA if max(crop.shape)>TARGET else cv2.INTER_CUBIC
    crop=cv2.resize(crop,(TARGET,TARGET),interpolation=interp)
    cm=(cv2.resize(cm,(TARGET,TARGET),interpolation=cv2.INTER_NEAREST)>60).astype(np.uint8)*255
    stem=os.path.basename(str(r["img"])).replace("_img.png","")
    ip=os.path.join(OUT,stem+"_img.png"); mkp=os.path.join(OUT,stem+"_msk.png")
    cv2.imwrite(ip,crop); cv2.imwrite(mkp,cm)
    d=r.to_dict(); d["img"]=ip; d["msk"]=mkp; rows.append(d); done+=1
    if done%400==0: print("  "+str(done))

new=pd.DataFrame(rows); new.to_csv(os.path.join(D,"cbis_mass_fixed.csv"),index=False)
print("\ndone "+str(done)+"  skipped "+str(skip))
# verify
mf=[]
for _,r in new.sample(min(100,len(new)),random_state=0).iterrows():
    mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    if mk is not None: mf.append((mk>127).mean())
print("NEW mass crop mask coverage: median "+format(100*np.median(mf),".0f")+"%  (want ~15-40%)")
print("label balance:", dict(new["label"].value_counts()))
print("saved cbis_mass_fixed.csv + crops_fixed_mass/")

  400
  800
  1200
  1600

done 1696  skipped 0
NEW mass crop mask coverage: median 22%  (want ~15-40%)
label balance: {0: np.int64(912), 1: np.int64(784)}
saved cbis_mass_fixed.csv + crops_fixed_mass/


**`BCF` cell 335** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [4]:
# ══════════════════════════════════════════════════════════════════════
# INSPECT + FIX mass mask fragmentation
#   diagnose: is the mask one lesion with ragged edges, or truly fragmented?
#   fix: keep the largest connected component (the mass) + small halo
#   shows before/after so you can SEE it
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"; FIG=os.path.join(D,"figures","mass_check")
S=512; _clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
df=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv"))

# ---- diagnose: for multi-piece masks, how big is the biggest piece? ----
print("diagnosing fragmentation...")
big_frac=[]
for _,r in df.iterrows():
    mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    if mk is None: continue
    m=(mk>127).astype(np.uint8)
    n,lab,st,_=cv2.connectedComponentsWithStats(m,8)
    if n<=1: big_frac.append(1.0); continue
    sizes=sorted([st[i,cv2.CC_STAT_AREA] for i in range(1,n)],reverse=True)
    big_frac.append(sizes[0]/max(sum(sizes),1))
big_frac=np.array(big_frac)
print("  biggest piece as % of total mask:")
print("    median "+format(100*np.median(big_frac),".0f")+"%")
print("    masks where biggest piece is >90% of mask: "+format(100*(big_frac>0.9).mean(),".0f")+"%")
print("    -> if most are >90%, it's ONE lesion + tiny ragged bits (harmless, easy fix)")
print("    -> if many are <70%, masks are truly split (needs real fix)\n")

# ---- FIX: keep largest component + close small gaps ----
def clean_mass_mask(mk):
    m=(mk>127).astype(np.uint8)
    # close small gaps first (merges near pieces)
    m=cv2.morphologyEx(m,cv2.MORPH_CLOSE,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9)))
    n,lab,st,_=cv2.connectedComponentsWithStats(m,8)
    if n<=1: return m*255
    big=1+int(np.argmax(st[1:,cv2.CC_STAT_AREA]))
    keep=(lab==big).astype(np.uint8)
    # fill holes inside the lesion
    keep=cv2.morphologyEx(keep,cv2.MORPH_CLOSE,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(15,15)))
    return keep*255

# apply + backup originals
viz=[]; before_pieces=[]; after_pieces=[]
for _,r in df.iterrows():
    mk=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    if mk is None: continue
    b=cv2.connectedComponents((mk>127).astype(np.uint8))[0]-1
    fixed=clean_mass_mask(mk)
    a=cv2.connectedComponents((fixed>127).astype(np.uint8))[0]-1
    before_pieces.append(b); after_pieces.append(a)
    cv2.imwrite(r["msk"].replace("_msk.png","_msk_orig.png"),mk)  # backup
    cv2.imwrite(r["msk"],fixed)
    if len(viz)<6 and b>1: viz.append((r["img"],mk,fixed,r["label"]))

print("mask pieces: before median "+str(int(np.median(before_pieces)))+
      " -> after median "+str(int(np.median(after_pieces))))

# before/after figure
if viz:
    fig,ax=plt.subplots(len(viz),3,figsize=(11,3.4*len(viz)))
    if len(viz)==1: ax=ax.reshape(1,3)
    for k,(ip,old,new,lab) in enumerate(viz):
        im=cv2.resize(cv2.imread(ip,cv2.IMREAD_GRAYSCALE),(S,S)); g=_clahe.apply(im)
        oldm=(cv2.resize(old,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
        newm=(cv2.resize(new,(S,S),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
        a=cv2.cvtColor(g,cv2.COLOR_GRAY2RGB); b=cv2.cvtColor(g,cv2.COLOR_GRAY2RGB)
        c1,_=cv2.findContours(oldm,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE); cv2.drawContours(a,c1,-1,(255,120,0),2)
        c2,_=cv2.findContours(newm,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE); cv2.drawContours(b,c2,-1,(0,255,0),2)
        lbl="MALIGNANT" if lab==1 else "benign"
        for c,(img,t) in enumerate([(g,lbl),(a,"BEFORE ("+str(len(c1))+" pieces)"),(b,"AFTER ("+str(len(c2))+" piece)")]):
            ax[k,c].imshow(img,cmap="gray" if c==0 else None); ax[k,c].set_title(t,fontsize=9); ax[k,c].axis("off")
    plt.suptitle("Mass mask cleanup: keep largest component + fill",fontsize=11)
    plt.tight_layout(); p=os.path.join(FIG,"mass_mask_fixed.png")
    plt.savefig(p,dpi=130,bbox_inches="tight"); plt.close(); print("saved "+p)
print("done — originals backed up as *_msk_orig.png")

diagnosing fragmentation...
  biggest piece as % of total mask:
    median 100%
    masks where biggest piece is >90% of mask: 100%
    -> if most are >90%, it's ONE lesion + tiny ragged bits (harmless, easy fix)
    -> if many are <70%, masks are truly split (needs real fix)

mask pieces: before median 3 -> after median 1
saved /root/autodl-tmp/CBIS/figures/mass_check/mass_mask_fixed.png
done — originals backed up as *_msk_orig.png


## C · Official split verification


**`BCF` cell 279** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [3]:
# ══════════════════════════════════════════════════════════════════════
# SPLIT AUDIT — does our split match the OFFICIAL CBIS-DDSM split?
#   and: is the TEST set harder than the TRAIN set? (your fairness question)
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
D="/root/autodl-tmp/CBIS"
raw=pd.read_csv(os.path.join(D,"cbis_final.csv"))

print("="*74)
print("1. DOES OUR 'split' MATCH 'official_split'?")
print("="*74)
print("  official_split values: "+str(dict(raw.official_split.value_counts())))
print("  our    split values: "+str(dict(raw.split.value_counts())))
print()
# normalise for comparison (official is usually train/test only)
o=raw.official_split.astype(str).str.lower().str.strip()
s=raw.split.astype(str).str.lower().str.strip()
s_bin=s.replace({"val":"train"})            # our val came out of train
match=(o==s_bin)
print("  rows where our split (val folded into train) == official: "+
      str(int(match.sum()))+"/"+str(len(raw))+" = "+format(100*match.mean(),".1f")+"%")
print()
print("  cross-tab (rows = official, cols = ours):")
print(pd.crosstab(o,s).to_string())
print()
if match.mean()>0.97:
    print("  >> OUR SPLIT IS THE OFFICIAL SPLIT (val carved out of official train). GOOD.")
else:
    print("  >> MISMATCH. Our split is NOT the official one — this must be stated,")
    print("     or we should re-run using official_split.")

print("\n"+"="*74)
print("2. IS THE TEST SET HARDER THAN TRAIN?  (your fairness question)")
print("="*74)
for kind in ["mass","calcification"]:
    sub=raw[raw.abn_type==kind]
    print("\n "+kind)
    for col in ["assessment","subtlety","density"]:
        if col not in sub.columns: continue
        print("   "+col+":")
        for sp in ["train","val","test"]:
            g=sub[sub.split==sp]
            if len(g)==0: continue
            v=g[col].dropna()
            if len(v)==0: continue
            if col=="assessment":
                hard=100*(v==4).mean(); desc="% BI-RADS 4"
            elif col=="subtlety":
                hard=100*(v<=2).mean(); desc="% subtlety 1-2"
            else:
                hard=100*(v>=4).mean(); desc="% density 4"
            print("     "+sp.ljust(6)+"n="+str(len(g)).rjust(4)+
                  "  mean "+format(v.mean(),".2f")+
                  "   "+desc+" = "+format(hard,".0f")+"%")
    print("   malignant fraction:")
    for sp in ["train","val","test"]:
        g=sub[sub.split==sp]
        if len(g): print("     "+sp.ljust(6)+format(g.label.mean(),".3f"))
print("\n  >> if train and test show SIMILAR percentages, the model IS trained on")
print("     hard cases and the evaluation is fair. If test is much harder, that")
print("     is a real distribution shift and must be reported.")

1. DOES OUR 'split' MATCH 'official_split'?
  official_split values: {'train': np.int64(2858), 'test': np.int64(704)}
  our    split values: {'train': np.int64(2405), 'test': np.int64(704), 'val': np.int64(453)}

  rows where our split (val folded into train) == official: 3562/3562 = 100.0%

  cross-tab (rows = official, cols = ours):
split           test  train  val
official_split                  
test             704      0    0
train              0   2405  453

  >> OUR SPLIT IS THE OFFICIAL SPLIT (val carved out of official train). GOOD.

2. IS THE TEST SET HARDER THAN TRAIN?  (your fairness question)

 mass
   assessment:
     train n=1122  mean 3.47   % BI-RADS 4 = 39%
     val   n= 196  mean 3.69   % BI-RADS 4 = 51%
     test  n= 378  mean 3.53   % BI-RADS 4 = 45%
   subtlety:
     train n=1122  mean 3.97   % subtlety 1-2 = 10%
     val   n= 196  mean 3.92   % subtlety 1-2 = 15%
     test  n= 378  mean 3.79   % subtlety 1-2 = 15%
   density:
     train n=1122  mean 2.18   % den

**`BCF` cell 280** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [4]:
# ══════════════════════════════════════════════════════════════════════
# SPLIT AUDIT — compact, writes full report to a text file
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
D="/root/autodl-tmp/CBIS"
raw=pd.read_csv(os.path.join(D,"cbis_final.csv"))
OUT=os.path.join(D,"split_audit.txt")
L=[]
def say(s=""):
    L.append(str(s))

# ---- 1. official vs ours ----
say("="*70); say("1. OUR SPLIT vs OFFICIAL SPLIT"); say("="*70)
o=raw.official_split.astype(str).str.lower().str.strip()
s=raw.split.astype(str).str.lower().str.strip()
say("official values: "+str(dict(o.value_counts())))
say("ours     values: "+str(dict(s.value_counts())))
s_bin=s.replace({"val":"train"})
m=(o==s_bin)
say("")
say("agreement (our val folded back into train): "+
    str(int(m.sum()))+"/"+str(len(raw))+" = "+format(100*m.mean(),".1f")+"%")
say("")
say("crosstab (rows=official, cols=ours):")
say(pd.crosstab(o,s).to_string())
say("")
say("VERDICT: "+("OUR SPLIT = OFFICIAL SPLIT (val carved from official train)"
               if m.mean()>0.97 else "MISMATCH — our split is NOT the official one"))

# ---- 2. train vs test difficulty ----
say(""); say("="*70); say("2. IS TEST HARDER THAN TRAIN?"); say("="*70)
tab=[]
for kind in ["mass","calcification"]:
    sub=raw[raw.abn_type==kind]
    for sp in ["train","val","test"]:
        g=sub[sub.split==sp]
        if len(g)==0: continue
        row=dict(lesion=kind, split=sp, n=len(g),
                 pct_birads4=100*(g.assessment==4).mean() if "assessment" in g else np.nan,
                 mean_subtlety=g.subtlety.mean() if "subtlety" in g else np.nan,
                 pct_subtle_12=100*(g.subtlety<=2).mean() if "subtlety" in g else np.nan,
                 pct_density4=100*(g.density>=4).mean() if "density" in g else np.nan,
                 pct_malignant=100*g.label.mean())
        tab.append(row)
T=pd.DataFrame(tab).round(1)
say(T.to_string(index=False))
say("")
# explicit train-vs-test deltas
say("TRAIN -> TEST shift:")
for kind in ["mass","calcification"]:
    tr=T[(T.lesion==kind)&(T.split=="train")]; te=T[(T.lesion==kind)&(T.split=="test")]
    if len(tr)==0 or len(te)==0: continue
    tr=tr.iloc[0]; te=te.iloc[0]
    say("  "+kind)
    for c,nm in [("pct_birads4","BI-RADS 4 %"),("pct_subtle_12","subtlety 1-2 %"),
                 ("pct_density4","density 4 %"),("pct_malignant","malignant %")]:
        d=te[c]-tr[c]
        flag="  <-- TEST HARDER" if d>8 else ("  <-- test easier" if d<-8 else "")
        say("    "+nm.ljust(16)+"train "+format(tr[c],"5.1f")+"  test "+format(te[c],"5.1f")+
            "   delta "+format(d,"+.1f")+flag)
say("")
say("Deltas within +/-8 points = comparable difficulty = evaluation is FAIR.")

txt="\n".join(L)
open(OUT,"w").write(txt)
T.to_csv(os.path.join(D,"split_difficulty_table.csv"),index=False)
print(txt[:1500])
print("\n... FULL REPORT SAVED TO: "+OUT)
print("open it with:  print(open('"+OUT+"').read())")

1. OUR SPLIT vs OFFICIAL SPLIT
official values: {'train': np.int64(2858), 'test': np.int64(704)}
ours     values: {'train': np.int64(2405), 'test': np.int64(704), 'val': np.int64(453)}

agreement (our val folded back into train): 3562/3562 = 100.0%

crosstab (rows=official, cols=ours):
split           test  train  val
official_split                  
test             704      0    0
train              0   2405  453

VERDICT: OUR SPLIT = OFFICIAL SPLIT (val carved from official train)

2. IS TEST HARDER THAN TRAIN?
       lesion split    n  pct_birads4  mean_subtlety  pct_subtle_12  pct_density4  pct_malignant
         mass train 1122         38.6            4.0           10.2           7.4           47.5
         mass   val  196         51.0            3.9           14.8          13.3           53.1
         mass  test  378         44.7            3.8           14.6          11.6           38.9
calcification train 1283         50.3            3.3           23.1           0.0           

## D · Crop and mask evidence


**`BCF` cell 262** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════
# DOES THE MASK DRIVE THE CLASSIFIER?  (tests your hypothesis)
#   1. correlation: Dice vs probability   (whole test set, not just errors)
#   2. correlation: mask AREA vs probability   <- the likely shortcut
#   3. shape-features-ALONE logistic regression AUC
#      if geometry alone ~= full model, the classifier is keying on mask shape
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd, cv2, torch, torch.nn as nn
from scipy.stats import pearsonr, pointbiserialr
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
D="/root/autodl-tmp/CBIS"; DEV=torch.device("cuda" if torch.cuda.is_available() else "cpu"); IMG=256

df=pd.read_csv(os.path.join(D,"ensemble_test_perimage.csv"))
df=df[df.group=="CBIS mass"].reset_index(drop=True)     # focus where you saw it

# --- segmentation model ---
def cb(i,o):
    return nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True),
                         nn.Conv2d(o,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(True))
class AG(nn.Module):
    def __init__(s,g,x,i):
        super().__init__()
        s.Wg=nn.Sequential(nn.Conv2d(g,i,1),nn.BatchNorm2d(i)); s.Wx=nn.Sequential(nn.Conv2d(x,i,1),nn.BatchNorm2d(i))
        s.psi=nn.Sequential(nn.Conv2d(i,1,1),nn.BatchNorm2d(1),nn.Sigmoid()); s.r=nn.ReLU(True)
    def forward(s,g,x): return x*s.psi(s.r(s.Wg(g)+s.Wx(x)))
class AttnPlain(nn.Module):
    def __init__(s,b=32):
        super().__init__()
        s.e1=cb(1,b);s.e2=cb(b,b*2);s.e3=cb(b*2,b*4);s.e4=cb(b*4,b*8)
        s.p=nn.MaxPool2d(2); s.bn=cb(b*8,b*16)
        s.u4=nn.ConvTranspose2d(b*16,b*8,2,2); s.a4=AG(b*8,b*8,b*4); s.d4=cb(b*16,b*8)
        s.u3=nn.ConvTranspose2d(b*8,b*4,2,2);  s.a3=AG(b*4,b*4,b*2); s.d3=cb(b*8,b*4)
        s.u2=nn.ConvTranspose2d(b*4,b*2,2,2);  s.a2=AG(b*2,b*2,b);   s.d2=cb(b*4,b*2)
        s.u1=nn.ConvTranspose2d(b*2,b,2,2);    s.a1=AG(b,b,b//2);    s.d1=cb(b*2,b)
        s.out=nn.Conv2d(b,2,1)
    def forward(s,x):
        e1=s.e1(x);e2=s.e2(s.p(e1));e3=s.e3(s.p(e2));e4=s.e4(s.p(e3)); bo=s.bn(s.p(e4))
        g4=s.u4(bo); d4=s.d4(torch.cat([g4,s.a4(g4,e4)],1))
        g3=s.u3(d4); d3=s.d3(torch.cat([g3,s.a3(g3,e3)],1))
        g2=s.u2(d3); d2=s.d2(torch.cat([g2,s.a2(g2,e2)],1))
        g1=s.u1(d2); d1=s.d1(torch.cat([g1,s.a1(g1,e1)],1))
        return s.out(d1)
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
seg=None
for f in ["seg_cbis_mass.pth","ablate_plain_cbis_mass.pth","seg_ds_cbis_mass.pth"]:
    p=os.path.join(D,f)
    if os.path.exists(p):
        seg=AttnPlain().to(DEV); sd=torch.load(p,map_location=DEV)
        sd=sd.get("state_dict",sd) if isinstance(sd,dict) else sd
        try: seg.load_state_dict(sd); seg.eval(); break
        except Exception: seg=None
@torch.no_grad()
def pmask(img):
    x=cv2.resize(_clahe.apply(img),(256,256))
    t=torch.from_numpy(x.astype(np.float32)/255.).unsqueeze(0).unsqueeze(0).to(DEV)
    with torch.amp.autocast(device_type="cuda"):
        return (torch.softmax(seg(t).float(),1)[0,1]>0.5).cpu().numpy().astype(np.uint8)

def shape_feats(m):
    m=(m>0).astype(np.uint8)
    if m.sum()<10: return np.zeros(6,np.float32)
    c,_=cv2.findContours(m,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    if not c: return np.zeros(6,np.float32)
    c=max(c,key=cv2.contourArea); area=cv2.contourArea(c); per=cv2.arcLength(c,True)+1e-6
    h=cv2.convexHull(c); ha=cv2.contourArea(h)+1e-6; x,y,w,hh=cv2.boundingRect(c)
    return np.array([4*np.pi*area/(per*per),area/ha,area/(w*hh+1e-6),w/(hh+1e-6),m.mean(),per/(area+1e-6)],np.float32)

rows=[]
for _,r in df.iterrows():
    img=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
    gtr=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE) if isinstance(r["msk"],str) and os.path.exists(str(r["msk"])) else None
    if img is None: continue
    img=cv2.resize(img,(256,256)); pr=pmask(img)
    gt=(cv2.resize(gtr,(256,256),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8) if gtr is not None else np.zeros((256,256),np.uint8)
    dice=(2*(pr*gt).sum()+1)/(pr.sum()+gt.sum()+1)
    f=shape_feats(pr)
    rows.append(dict(patient_id=r["patient_id"], prob=r["prob"], true=r["true"],
                     dice=dice, pred_area=float(pr.mean()), gt_area=float(gt.mean()),
                     circ=f[0], solidity=f[1], extent=f[2], aspect=f[3], fill=f[4], perim_ratio=f[5]))
A=pd.DataFrame(rows)
A.to_csv(os.path.join(D,"mask_vs_classification_diagnostic.csv"),index=False)
print("n = "+str(len(A))+"  (CBIS mass test)\n")

print("="*64)
print("1. DOES DICE PREDICT THE CLASSIFIER'S PROBABILITY?")
print("="*64)
r,p=pearsonr(A.dice,A.prob)
print("   corr(Dice, prob)      = "+format(r,"+.3f")+"   p="+format(p,".2e"))
r2,p2=pearsonr(A.pred_area,A.prob)
print("   corr(mask AREA, prob) = "+format(r2,"+.3f")+"   p="+format(p2,".2e")+"   <- the suspected shortcut")
r3,_=pointbiserialr(A.true,A.dice)
print("   corr(true label, Dice)= "+format(r3,"+.3f")+"   (is Dice itself higher for malignant?)")
r4,_=pointbiserialr(A.true,A.pred_area)
print("   corr(true label, area)= "+format(r4,"+.3f")+"   (are malignant masses genuinely bigger?)")

print("\n"+"="*64)
print("2. MEAN VALUES BY OUTCOME")
print("="*64)
A["pred"]=(A.prob>0.5).astype(int)
A["kind"]=np.where((A.true==1)&(A.pred==1),"TP",np.where((A.true==0)&(A.pred==0),"TN",
           np.where((A.true==0)&(A.pred==1),"FP","FN")))
print(A.groupby("kind")[["prob","dice","pred_area"]].mean().round(3).to_string())

print("\n"+"="*64)
print("3. SHAPE FEATURES ALONE (no image) -> how much can geometry explain?")
print("="*64)
X=A[["circ","solidity","extent","aspect","fill","perim_ratio","pred_area"]].values
y=A.true.values; groups=A.patient_id.values
oof=np.zeros(len(y))
for tri,tei in GroupKFold(n_splits=5).split(X,y,groups):
    lr=LogisticRegression(max_iter=2000,class_weight="balanced").fit(X[tri],y[tri])
    oof[tei]=lr.predict_proba(X[tei])[:,1]
print("   shape-features-only AUC : "+format(roc_auc_score(y,oof),".4f"))
print("   your full deep model AUC: "+format(roc_auc_score(A.true,A.prob),".4f"))
print("\n   READ IT:")
print("   - if shape-only AUC is CLOSE to the deep model -> the classifier is mostly")
print("     using mask geometry, and the images add little (your hypothesis is right)")
print("   - if shape-only is much LOWER -> the deep features carry the signal,")
print("     and the mask is not the bottleneck")
print("="*64)

n = 378  (CBIS mass test)

1. DOES DICE PREDICT THE CLASSIFIER'S PROBABILITY?
   corr(Dice, prob)      = +0.140   p=6.27e-03
   corr(mask AREA, prob) = +0.033   p=5.25e-01   <- the suspected shortcut
   corr(true label, Dice)= +0.093   (is Dice itself higher for malignant?)
   corr(true label, area)= +0.077   (are malignant masses genuinely bigger?)

2. MEAN VALUES BY OUTCOME
       prob   dice  pred_area
kind                         
FN    0.424  0.863      0.412
FP    0.578  0.923      0.367
TN    0.457  0.887      0.352
TP    0.635  0.928      0.371

3. SHAPE FEATURES ALONE (no image) -> how much can geometry explain?
   shape-features-only AUC : 0.5791
   your full deep model AUC: 0.7683

   READ IT:
   - if shape-only AUC is CLOSE to the deep model -> the classifier is mostly
     using mask geometry, and the images add little (your hypothesis is right)
   - if shape-only is much LOWER -> the deep features carry the signal,
     and the mask is not the bottleneck


**`BCF` cell 277** — ══════════════════════════════════════════════════════════════════════  
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════
# CRITICAL CHECK — are the crops just the mask bounding box?
#   1. mask area as fraction of crop area
#   2. is the mask centred and filling the crop? (crop==bbox artifact)
#   3. does the mask sit on the brightest structures, or arbitrary tissue?
#   4. visualise worst offenders
# ══════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
D="/root/autodl-tmp/CBIS"; FIG=os.path.join(D,"figures"); os.makedirs(FIG,exist_ok=True)
_clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))

raw=pd.read_csv(os.path.join(D,"cbis_final.csv"))
rows=[]
for _,r in raw.iterrows():
    m=cv2.imread(r["msk"],cv2.IMREAD_GRAYSCALE)
    im=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE)
    if m is None or im is None: continue
    b=(m>127).astype(np.uint8)
    H,W=b.shape
    if b.sum()==0:
        rows.append(dict(img=r["img"],abn=r["abn_type"],label=r["label"],
                         frac=0,bbox_frac=0,cx_off=np.nan,cy_off=np.nan,
                         bright_in=np.nan,bright_out=np.nan,contrast=np.nan)); continue
    frac=b.mean()                                   # mask area / crop area
    ys,xs=np.where(b>0)
    bw=(xs.max()-xs.min()+1); bh=(ys.max()-ys.min()+1)
    bbox_frac=(bw*bh)/(W*H)                         # bbox area / crop area
    cx=(xs.mean()/W)-0.5; cy=(ys.mean()/H)-0.5      # centre offset (0 = perfectly centred)
    g=_clahe.apply(cv2.resize(im,(W,H))) if im.shape!=b.shape else _clahe.apply(im)
    bi=float(g[b>0].mean()); bo=float(g[b==0].mean()) if (b==0).sum()>0 else np.nan
    rows.append(dict(img=r["img"],abn=r["abn_type"],label=r["label"],
                     frac=float(frac),bbox_frac=float(bbox_frac),
                     cx_off=float(cx),cy_off=float(cy),
                     bright_in=bi,bright_out=bo,contrast=bi-bo))
A=pd.DataFrame(rows)
A.to_csv(os.path.join(D,"mask_geometry_audit.csv"),index=False)

print("="*76)
print("1. HOW MUCH OF THE CROP IS MASK?")
print("="*76)
for k,g in A.groupby("abn"):
    print("  "+k.ljust(15)+"n="+str(len(g)).rjust(4)+
          " | mask/crop area: mean "+format(g.frac.mean(),".3f")+
          "  median "+format(g.frac.median(),".3f")+
          "  >50%: "+format(100*(g.frac>0.5).mean(),".0f")+"%"+
          "  >70%: "+format(100*(g.frac>0.7).mean(),".0f")+"%")
print("\n  bbox/crop area (1.0 = crop IS the bounding box):")
for k,g in A.groupby("abn"):
    print("  "+k.ljust(15)+"mean "+format(g.bbox_frac.mean(),".3f")+
          "  median "+format(g.bbox_frac.median(),".3f")+
          "  >0.90: "+format(100*(g.bbox_frac>0.90).mean(),".0f")+"%")
print("\n  >> if bbox/crop is near 1.0 for most images, the crop was made from")
print("     the mask bounding box -> segmentation Dice is INFLATED and the mask")
print("     carries no localisation information for classification.")

print("\n"+"="*76)
print("2. IS THE MASK CENTRED? (crop==bbox artifact)")
print("="*76)
for k,g in A.groupby("abn"):
    print("  "+k.ljust(15)+"centre offset |x| mean "+format(g.cx_off.abs().mean(),".3f")+
          "  |y| mean "+format(g.cy_off.abs().mean(),".3f")+
          "   (0.00 = always dead-centre = suspicious)")

print("\n"+"="*76)
print("3. DOES THE MASK SIT ON BRIGHT STRUCTURE?")
print("="*76)
for k,g in A.groupby("abn"):
    print("  "+k.ljust(15)+"mean brightness inside "+format(g.bright_in.mean(),".1f")+
          " vs outside "+format(g.bright_out.mean(),".1f")+
          "  -> contrast "+format(g.contrast.mean(),"+.1f"))
    print("                 lesions where inside is DARKER than outside: "+
          format(100*(g.contrast<0).mean(),".0f")+"%")
print("\n  >> a mask that is not brighter than its surroundings may be")
print("     mislocated, or the lesion may be genuinely low-contrast.")

# ---- 4. visualise the worst offenders ----
def show(sel,title,fname,n=6):
    s=sel.head(n).reset_index(drop=True)
    if len(s)==0: print("none for "+title); return
    fig,ax=plt.subplots(len(s),3,figsize=(10,3.1*len(s)))
    if len(s)==1: ax=ax.reshape(1,3)
    for k,r in s.iterrows():
        im=cv2.imread(r["img"],cv2.IMREAD_GRAYSCALE); m=cv2.imread(r["img"].replace("_img","_msk"),cv2.IMREAD_GRAYSCALE)
        row=raw[raw.img==r["img"]]
        if m is None and len(row): m=cv2.imread(row.iloc[0]["msk"],cv2.IMREAD_GRAYSCALE)
        if im is None or m is None: continue
        b=(cv2.resize(m,(im.shape[1],im.shape[0]),interpolation=cv2.INTER_NEAREST)>127).astype(np.uint8)
        ov=cv2.cvtColor(_clahe.apply(im),cv2.COLOR_GRAY2RGB)
        c,_=cv2.findContours(b,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(ov,c,-1,(0,220,255),2)
        for j,(x,t,cm) in enumerate([(_clahe.apply(im),os.path.basename(r["img"])[:18],"gray"),
                                     (b*255,"mask covers "+format(r["frac"]*100,".0f")+"% of crop","gray"),
                                     (ov,"contrast "+format(r["contrast"],"+.1f"),None)]):
            ax[k,j].imshow(x,cmap=cm) if cm else ax[k,j].imshow(x)
            ax[k,j].set_title(t,fontsize=8); ax[k,j].axis("off")
    plt.suptitle(title,fontsize=11); plt.tight_layout()
    p=os.path.join(FIG,fname); plt.savefig(p,dpi=140,bbox_inches="tight"); plt.close(); print("saved "+p)

calc=A[A.abn=="calcification"]
show(calc.sort_values("frac",ascending=False),"CALC — masks covering MOST of the crop","audit_calc_bigmask.png")
show(calc[calc.contrast<0].sort_values("contrast"),"CALC — mask DARKER than surroundings (possible mislocation)","audit_calc_darkmask.png")
print("\nsaved mask_geometry_audit.csv")

1. HOW MUCH OF THE CROP IS MASK?
  calcification  n=1866 | mask/crop area: mean 0.424  median 0.444  >50%: 7%  >70%: 0%
  mass           n=1696 | mask/crop area: mean 0.380  median 0.380  >50%: 1%  >70%: 0%

  bbox/crop area (1.0 = crop IS the bounding box):
  calcification  mean 0.602  median 0.592  >0.90: 0%
  mass           mean 0.595  median 0.592  >0.90: 0%

  >> if bbox/crop is near 1.0 for most images, the crop was made from
     the mask bounding box -> segmentation Dice is INFLATED and the mask
     carries no localisation information for classification.

2. IS THE MASK CENTRED? (crop==bbox artifact)
  calcification  centre offset |x| mean 0.023  |y| mean 0.018   (0.00 = always dead-centre = suspicious)
  mass           centre offset |x| mean 0.020  |y| mean 0.015   (0.00 = always dead-centre = suspicious)

3. DOES THE MASK SIT ON BRIGHT STRUCTURE?
  calcification  mean brightness inside 140.2 vs outside 128.8  -> contrast +11.4
                 lesions where inside is DARKER 